# TFM - Predicción de cancelaciones
## 1. Predicción de Cancelaciones de Reservas de Hotel

**Autor:** Grupo 4  
**Fecha:** 2026-02-18

---

### 1.1. Objetivo
Realizar un análisis exploratorio de los datos para:
- Conocer las dimensiones y estructura del dataset
- Entender la distribución de reservas por hotel
- Identificar cuáles hoteles son candidatos óptimos para el modelo predictivo
- Analizar el comportamiento de la variable objetivo (cancelaciones)
- Generar un modelo de predicción

## 1\. Diccionario de Variables

| Variable | Tipo | Descripcion |
|----------|------|-------------|
| **ID_RESERVA** | Int | Identificador unico de la reserva |
| **ID_HOTEL** | Int | Identificador del hotel |
| **LLEGADA** | DateTime | Fecha de check-in |
| **SALIDA** | DateTime | Fecha de check-out |
| **NOCHES** | Int | Numero de noches de alojamiento |
| **REGIMEN** | String | Tipo de regimen (All Inclusive, etc.) |
| **ID_TIPO** | Int | ID del tipo de habitacion |
| **TIPO** | String | Codigo del tipo de habitacion |
| **USO** | Int | Uso de la habitacion |
| **PAX** | Int | Numero total de huespedes |
| **ADULTOS** | Int | Numero de adultos |
| **NENES** | Int | Numero de ninos (2-13 anos) |
| **BEBES** | Int | Numero de bebes (<2 anos) |
| **ID_CLIENTE** | Int | Identificador del cliente |
| **TIPO_CLIENTE** | Int | Tipo de cliente |
| **CLIENTE** | String | Nombre del cliente/agencia |
| **ID_MULTIPLE** | Int | ID > 0 indica reserva de grupo |
| **ID_MONEDA** | Int | ID de la moneda |
| **MONEDA** | String | Moneda de pago |
| **STATUS** | Int | Estado: 1-2=OK, 3-4=Cancelada, 9=Especial |
| **ID_MOTIVO** | Int | ID del motivo de cancelacion |
| **ID_CARDEX** | Int | ID del cardex del cliente |
| **SUPLETORIA** | Int | Cama supletoria |
| **CUNAS** | Int | Numero de cunas |
| **FECHA_TOMA** | DateTime | Fecha en que se realizo la reserva |
| **FECHA_MOD** | DateTime | Fecha de ultima modificacion |
| **FECHA_CANCELACION** | DateTime | Fecha de cancelacion (si aplica) |
| **ID_FIDELIDAD** | Int | ID del programa de fidelizacion |
| **FIDELIDAD** | String | Programa de fidelizacion |
| **VALHAB** | Float | Produccion por habitacion |
| **VALPEN** | Float | Produccion por pension |
| **VALSERV** | Float | Produccion por servicios |
| **VALFIJOS** | Float | Produccion fija |
| **COMERCIALIZADORA** | Int | Booleano: 1 si es comercializadora |
| **CMVALHAB** | Float | Valor habitacion comercializadora |
| **CMVALPEN** | Float | Valor pension comercializadora |
| **CMCVALSERV** | Float | Valor servicios comercializadora |
| **AUTORIZO** | Int | Indicador de autorizacion |
| **GRATIS** | Int | Indicador de reserva gratuita |
| **PAIS** | String | Pais de origen del cliente |
| **SEGMENTO** | String | Segmento de mercado |
| **FUENTE_NEGOCIO** | String | Canal de venta |

> **Variable Target**: `STATUS = 3 o 4` indica **reserva cancelada**


---
## 2\. Configuracion del Entorno


In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Importar librerias necesarias
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

### 2.1. Paleta de Colores Corporativos

Colores oficiales de Aurea Hotel Group para las visualizaciones:


In [ ]:
# Paleta de colores corporativos Aurea Hotel Group
AUREA_COLORS = {
    'raw_sienna': '#D28B57',      # Color primario (dorado cobrizo)
    'luxor_gold': '#99782C',      # Color secundario (dorado oscuro)
    'spicy_mustard': '#6E550C',   # Color terciario (mostaza)
    'regent_blue': '#A1C0E4',     # Color complementario (azul suave)
    'cod_gray': '#1B1A1A',        # Color oscuro (gris carbon)
    'white': '#FFFFFF'            # Color claro
}

# Lista ordenada para usar en graficos
PALETA_AUREA = [
    '#D28B57',  # Raw Sienna
    '#99782C',  # Luxor Gold
    '#6E550C',  # Spicy Mustard
    '#A1C0E4',  # Regent Blue
    '#1B1A1A',  # Cod Gray
]

# Colores para variable binaria (cancelado / no cancelado)
COLOR_CANCELADO = '#D28B57'      # Raw Sienna
COLOR_NO_CANCELADO = '#A1C0E4'   # Regent Blue


---
## 3\. Carga de Datos


In [ ]:
BASE_PATH = '/content/drive/MyDrive/TFM_Aurea_Team/01_data/raw'

print(f'Leyendo archivos desde: {BASE_PATH}')


In [ ]:
# -----------------------------------------------
# 2.1 Cargamos archivo de reservas
# -----------------------------------------------

try:
    file_path = os.path.join(BASE_PATH, 'Reservas_22_23.csv')
    df_reservas = pd.read_csv(file_path, sep=';')
    print(f'[OK] Reservas cargado con exito: {df_reservas.shape[0]} filas.')
except Exception as e:
    print(f'[ERROR] No se pudo cargar Reservas. Razon: {e}')


In [ ]:
# -----------------------------------------------
# 2.2 Cargamos catalogos
# -----------------------------------------------

# Cargamos Hoteles
try:
    df_hoteles = pd.read_excel(os.path.join(BASE_PATH, 'Hoteles.xlsx'))
    print(f'[OK] Hoteles cargado con exito: {df_hoteles.shape[0]} filas.')
except Exception as e:
    print(f'[AVISO] No se pudo cargar Hoteles. Razon: {e}')

# Cargamos Habitaciones
try:
    df_habitaciones = pd.read_excel(os.path.join(BASE_PATH, 'CAT_HABITACIONES.xlsx'))
    print(f'[OK] Habitaciones cargado con exito: {df_habitaciones.shape[0]} filas.')
except Exception as e:
    print(f'[AVISO] No se pudo cargar Habitaciones. Razon: {e}')

# Cargamos Socios Rewards
try:
    df_rewards = pd.read_csv(os.path.join(BASE_PATH, 'num_socio_rewards(in).csv'), sep=';')
    print(f'[OK] Socios Rewards cargado con exito: {df_rewards.shape[0]} filas.')
except Exception as e:
    print(f'[AVISO] No se pudo cargar Socios Rewards. Razon: {e}')


---
## 4\. EDA Express


In [ ]:
# ===============================================
# 3. EDA EXPRESS
# ===============================================

print('\n----------------- 1. ESTRUCTURA DEL DATASET -----------------')
df_reservas.info()

print('\n\n----------------- 2. DETECCION DE NULOS -----------------')
nulls = df_reservas.isnull().sum()
print(nulls[nulls>0])

print('\n\n----------------- 3. ANALISIS DEL TARGET -----------------')
print(df_reservas.groupby('STATUS')['FECHA_CANCELACION'].count())
print('\nPorcentaje de Status')
distribucion_target = (df_reservas['STATUS'].value_counts(normalize=True)*100).round(2)
print(distribucion_target)

print('\n\n----------------- 4. ESTADISTICAS RAPIDAS -----------------')
print('.describe de algunas variables numericas')
display(df_reservas[['NOCHES','PAX','ADULTOS','NENES','BEBES','VALHAB','VALPEN','VALSERV','VALFIJOS','CMVALHAB', 'CMVALPEN', 'CMCVALSERV']].describe().T)
print('\n.describe de algunas variables categoricas (Faltan booleanos y codigos)')
display(df_reservas.describe(include=object).T)


---
## 5\. Joins con Catalogos


In [ ]:
# ===============================================
# 4. JOINS CON CATALOGOS
# ===============================================

# -----------------------------------------------
# 4.1 Normalizacion de TIPO para el cruce
# -----------------------------------------------
df_reservas['TIPO_NORMALIZADO'] = df_reservas['TIPO'].str.strip().str.upper()
df_habitaciones['NOMBRE_HABITACION_NORM'] = df_habitaciones['NOMBRE_HABITACION'].str.strip().str.upper()

# -----------------------------------------------
# 4.2 Join con catalogo de Hoteles
# -----------------------------------------------
df_reservas = df_reservas.merge(
    df_hoteles[['ID_HOTEL', 'NOMBRE_HOTEL']],
    on='ID_HOTEL',
    how='left'
)
print(f'[OK] Join con Hoteles. Sin match: {df_reservas["NOMBRE_HOTEL"].isna().sum()}')

# -----------------------------------------------
# 4.3 Join con catalogo de Habitaciones
# -----------------------------------------------
df_reservas = df_reservas.merge(
    df_habitaciones[['NOMBRE_HABITACION_NORM', 'NOMBRE_HABITACION', 'Categoria']],
    left_on='TIPO_NORMALIZADO',
    right_on='NOMBRE_HABITACION_NORM',
    how='left'
)
df_reservas = df_reservas.drop(columns=['NOMBRE_HABITACION_NORM'])
print(f'[OK] Join con Habitaciones. Sin match: {df_reservas["NOMBRE_HABITACION"].isna().sum()}')

# -----------------------------------------------
# 4.4 Join con Socios Rewards
# -----------------------------------------------
df_reservas = df_reservas.merge(
    df_rewards[['ID_RESERVA', 'NUMSOCIO']],
    on='ID_RESERVA',
    how='left'
)
print(f'[OK] Join con Rewards. Con socio: {df_reservas["NUMSOCIO"].notna().sum()}')

# -----------------------------------------------
# 4.5 Verificacion
# -----------------------------------------------
print(f'\nDataset final: {df_reservas.shape}')
df_reservas.info()


### (auxiliar para guardar raw)

In [ ]:
# ruta = BASE_PATH + '/df_reservas_join.csv'
# df_reservas.to_csv(ruta, index=False)

### (auxiliar para recargar raw)

In [ ]:
# def leer_y_crear_parquet(ruta):
#   # Intentar cargar el parquet si lo hay
#   try:
#     nombre = ruta.rsplit('/',1 )[1]   #reverse split
#     parquet_ruta = ruta.replace(nombre, 'parquet/'+nombre)
#     parquet_ruta = parquet_ruta.replace('.csv', '.parquet')
#     df = pd.read_parquet(parquet_ruta)
#   except:
#     # Si no existe el parquet lo creamos para la próxima
#     df = pd.read_csv(ruta, low_memory=False)
#     df.to_parquet(parquet_ruta, index=False)
#   return df

# ruta = BASE_PATH + '/df_reservas_join.csv'
# df_reservas = leer_y_crear_parquet(ruta)
# df_reservas.shape

---
## 6\. Conversion de fechas


In [ ]:
# -----------------------------------------------
# Conversion de fechas
# -----------------------------------------------
cols_fechas = ['LLEGADA', 'SALIDA', 'FECHA_TOMA', 'FECHA_MOD', 'FECHA_CANCELACION']
for col in cols_fechas:
    df_reservas[col] = pd.to_datetime(df_reservas[col], errors='coerce')

print('[OK] Fechas convertidas a datetime')

---
## 7\. Analisis Detallado de Nulos (Subset Filtrado)

Analizamos la presencia de valores nulos en las variables categoricas clave (`REGIMEN`, `CLIENTE`) para el subset de hoteles seleccionados.


In [ ]:
# Imprimimos la lista de Nulls actuales en el dataset filtrado
nulls = df_reservas.isnull().sum()
print(nulls[nulls>0])


In [ ]:
# -------------------------------------------------------------------------
# Analisis de Nulos: REGIMEN
# -------------------------------------------------------------------------

# Creamos un dataset filtrando solo los nulos de REGIMEN
nulls_regimen = df_reservas[df_reservas['REGIMEN'].isnull()]

if len(nulls_regimen) > 0:
    print(f"Se encontraron {len(nulls_regimen)} registros con REGIMEN nulo.")
    # Mostramos los nulos con columnas de contexto
    print(nulls_regimen[['NOMBRE_HOTEL', 'NOCHES', 'STATUS', 'PAX', 'COMERCIALIZADORA', 'VALPEN', 'VALHAB', 'GRATIS']])

    # Distribucion por hotel
    print("\nDistribución por Hotel:")
    print(nulls_regimen['NOMBRE_HOTEL'].value_counts(normalize=True)*100)
else:
    print("[OK] No hay nulos en REGIMEN para los hoteles seleccionados.")


In [ ]:
# -------------------------------------------------------------------------
# Analisis de Nulos: CLIENTE
# -------------------------------------------------------------------------

# Creamos un dataset filtrando solo los nulos de CLIENTE
nulls_cliente = df_reservas[df_reservas['CLIENTE'].isnull()]

if len(nulls_cliente) > 0:
    print(f"Se encontraron {len(nulls_cliente)} registros con CLIENTE nulo.")
    # Mostramos los nulos con columnas de contexto
    print(nulls_cliente[['NOMBRE_HOTEL', 'NOCHES', 'STATUS', 'PAX', 'COMERCIALIZADORA', 'VALPEN', 'VALHAB', 'GRATIS']])
else:
    print("[OK] No hay nulos en CLIENTE para los hoteles seleccionados.")


In [ ]:
# -------------------------------------------------------------------------
# Verificacion: NOMBRE_HOTEL
# -------------------------------------------------------------------------

# Verificamos si existen nulos en NOMBRE_HOTEL dentro del subset filtrado
nulls_hoteles = df_reservas[df_reservas['NOMBRE_HOTEL'].isnull()]

if len(nulls_hoteles) > 0:
    print(f"[ATENCION] Se encontraron {len(nulls_hoteles)} registros con NOMBRE_HOTEL nulo.")
    print(nulls_hoteles[['ID_HOTEL', 'NOMBRE_HOTEL', 'STATUS']])
    print("\nIDs afectados:")
    print(nulls_hoteles['ID_HOTEL'].value_counts())
else:
    print("[OK] Integridad Correcta: No hay nulos en NOMBRE_HOTEL para el subset seleccionado (IDs 96, 106, 107).")


---
## 8\. EDA - Seleccion de Hoteles

En esta seccion analizaremos la distribucion de reservas por hotel para:
1. Identificar cuales son los hoteles mas representativos del dataset
2. Analizar sus tasas de cancelacion
3. Verificar su diversidad de mercados (paises) y segmentos

El objetivo es seleccionar un **subset de hoteles** que nos permita entrenar modelos robustos sin perder generalidad.


In [ ]:
# Ver cuántos hoteles hay
hoteles_unicos = df_reservas[['ID_HOTEL', 'NOMBRE_HOTEL']].drop_duplicates().sort_values('ID_HOTEL')
print(f'Número de hoteles en el dataset: {len(hoteles_unicos)}')
print('\nListado de hoteles:')
print('-' * 60)
for idx, row in hoteles_unicos.iterrows():
    print(f"ID {row['ID_HOTEL']}: {row['NOMBRE_HOTEL']}")


In [ ]:
# Conteo de reservas por hotel
reservas_hotel = df_reservas.groupby(['ID_HOTEL', 'NOMBRE_HOTEL']).size().reset_index(name='NUM_RESERVAS')
reservas_hotel = reservas_hotel.sort_values('NUM_RESERVAS', ascending=False)

# Calcular porcentaje y acumulado
reservas_hotel['PORCENTAJE'] = (reservas_hotel['NUM_RESERVAS'] / reservas_hotel['NUM_RESERVAS'].sum() * 100).round(2)
reservas_hotel['ACUMULADO'] = reservas_hotel['PORCENTAJE'].cumsum().round(2)

# Crear etiqueta con ID + Nombre
reservas_hotel['ETIQUETA'] = reservas_hotel.apply(lambda x: f"{x['NOMBRE_HOTEL']} (ID: {x['ID_HOTEL']})", axis=1)

print('Distribución de reservas por hotel (ordenado de mayor a menor):')
print('=' * 90)
print(reservas_hotel[['ID_HOTEL', 'NOMBRE_HOTEL', 'NUM_RESERVAS', 'PORCENTAJE', 'ACUMULADO']].to_string(index=False))


In [ ]:
# Gráfico de barras: Volumen de reservas
fig, ax = plt.subplots(figsize=(12, 6))

datos_grafico = reservas_hotel.sort_values('NUM_RESERVAS', ascending=True)

# Usar escala de colores según volumen (Corporativo)
# AUREA_COLORS keys: 'raw_sienna', 'luxor_gold', 'spicy_mustard', etc.
colores = [AUREA_COLORS['spicy_mustard'] if i % 2 == 0 else AUREA_COLORS['raw_sienna'] for i in range(len(datos_grafico))]

barras = ax.barh(datos_grafico['ETIQUETA'], datos_grafico['NUM_RESERVAS'], color=colores)

for i, (idx, row) in enumerate(datos_grafico.iterrows()):
    ax.text(row['NUM_RESERVAS'] + 1000, i, f"{row['NUM_RESERVAS']:,} ({row['PORCENTAJE']}%)", va='center', fontsize=9)

ax.set_xlabel('Número de Reservas')
ax.set_title('Volumen de Reservas por Hotel')
ax.set_xlim(0, datos_grafico['NUM_RESERVAS'].max() * 1.25)

plt.tight_layout()
plt.show()


In [ ]:
# Gráfico de tarta: Top 8 hoteles + Otros
fig, ax = plt.subplots(figsize=(10, 8))

# Top 8 hoteles
top8 = reservas_hotel.head(8).copy()
otros_total = reservas_hotel.iloc[8:]['NUM_RESERVAS'].sum()

# Añadir categoria Otros si hay más de 8 hoteles
if otros_total > 0:
    otros_row = pd.DataFrame({'ETIQUETA': ['Otros hoteles'], 'NUM_RESERVAS': [otros_total]})
    datos_tarta = pd.concat([top8[['ETIQUETA', 'NUM_RESERVAS']], otros_row], ignore_index=True)
else:
    datos_tarta = top8[['ETIQUETA', 'NUM_RESERVAS']]

# Colores corporativos para la tarta
colores_tarta = [AUREA_COLORS['raw_sienna'], AUREA_COLORS['luxor_gold'], AUREA_COLORS['spicy_mustard'],
                 '#C69C6D', '#7F6A44', '#A0885C', '#E6CDB8', '#BFA88F', '#CCCCCC'] # Variaciones y gris para otros

ax.pie(datos_tarta['NUM_RESERVAS'], labels=datos_tarta['ETIQUETA'], autopct='%1.1f%%',
       colors=colores_tarta[:len(datos_tarta)], startangle=90)
ax.set_title('Distribución del Volumen de Reservas\n(Top 8 hoteles + Otros)')

plt.tight_layout()
plt.show()


In [ ]:
# Crear variable binaria de cancelación
df_reservas['CANCELADO'] = (df_reservas['STATUS'].isin([3,4])).astype(int)

# Calcular tasa de cancelación por hotel
cancelaciones_hotel = df_reservas.groupby(['ID_HOTEL', 'NOMBRE_HOTEL']).agg(
    TOTAL_RESERVAS=('ID_RESERVA', 'count'),
    CANCELADOS=('CANCELADO', 'sum')
).reset_index()

cancelaciones_hotel['TASA_CANCELACION'] = (cancelaciones_hotel['CANCELADOS'] / cancelaciones_hotel['TOTAL_RESERVAS'] * 100).round(2)
cancelaciones_hotel['CONFIRMADAS'] = cancelaciones_hotel['TOTAL_RESERVAS'] - cancelaciones_hotel['CANCELADOS']
cancelaciones_hotel['ETIQUETA'] = cancelaciones_hotel.apply(lambda x: f"{x['NOMBRE_HOTEL']} (ID: {x['ID_HOTEL']})", axis=1)
cancelaciones_hotel = cancelaciones_hotel.sort_values('TOTAL_RESERVAS', ascending=False)

print('Tasa de cancelación por hotel:')
print('=' * 100)
print(cancelaciones_hotel[['ID_HOTEL', 'NOMBRE_HOTEL', 'TOTAL_RESERVAS', 'CANCELADOS', 'CONFIRMADAS', 'TASA_CANCELACION']].to_string(index=False))


In [ ]:
# Media global de cancelaciones
media_global = df_reservas['CANCELADO'].mean() * 100
print(f'Tasa de cancelación global del dataset: {media_global:.2f}%')


In [ ]:
# Gráfico de barras: Tasa de cancelación por hotel
fig, ax = plt.subplots(figsize=(12, 6))

datos_grafico = cancelaciones_hotel.sort_values('TASA_CANCELACION', ascending=True)

colores = [AUREA_COLORS['spicy_mustard'] if x > media_global else AUREA_COLORS['luxor_gold']
           for x in datos_grafico['TASA_CANCELACION']]

barras = ax.barh(datos_grafico['ETIQUETA'], datos_grafico['TASA_CANCELACION'], color=colores)

for i, (idx, row) in enumerate(datos_grafico.iterrows()):
    ax.text(row['TASA_CANCELACION'] + 0.3, i, f"{row['TASA_CANCELACION']}%", va='center', fontsize=14)

ax.set_xlabel('Tasa de Cancelación (%)')
ax.tick_params(axis='y', labelsize=14)
ax.set_title('Tasa de Cancelación por Hotel')
ax.set_xlim(0, datos_grafico['TASA_CANCELACION'].max() * 1.15)

# Línea de referencia: media global
ax.axvline(x=media_global, color='black', linestyle='--', linewidth=2, label=f'Media global: {media_global:.2f}%')
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de barras apiladas: Canceladas vs Confirmadas por hotel
fig, ax = plt.subplots(figsize=(12, 6))

datos_grafico = cancelaciones_hotel.sort_values('TOTAL_RESERVAS', ascending=True)

ax.barh(datos_grafico['ETIQUETA'], datos_grafico['CONFIRMADAS'], label='Confirmadas', color=AUREA_COLORS['luxor_gold'])
ax.barh(datos_grafico['ETIQUETA'], datos_grafico['CANCELADOS'], left=datos_grafico['CONFIRMADAS'], label='Canceladas', color=AUREA_COLORS['raw_sienna'])

ax.set_xlabel('Número de Reservas')
ax.set_title('Reservas Confirmadas vs Canceladas por Hotel')
ax.legend()


plt.tight_layout()
plt.show()


In [ ]:
# Número de países únicos por hotel
paises_por_hotel = df_reservas.groupby(['ID_HOTEL', 'NOMBRE_HOTEL'])['PAIS'].nunique().reset_index(name='NUM_PAISES')
paises_por_hotel = paises_por_hotel.sort_values('NUM_PAISES', ascending=False)

print('Diversidad de países de origen por hotel:')
print('=' * 60)
print(paises_por_hotel.to_string(index=False))


In [ ]:
# Top 5 países por cada hotel (top 8 hoteles por volumen)
print('Top 5 países de origen por hotel:')
print('=' * 80)

for hotel_id in reservas_hotel['ID_HOTEL'].head(8).tolist():
    hotel_df = df_reservas[df_reservas['ID_HOTEL'] == hotel_id]
    hotel_nombre = hotel_df['NOMBRE_HOTEL'].iloc[0]
    top_paises = hotel_df['PAIS'].value_counts().head(5)

    print(f'\n{hotel_nombre} (ID: {hotel_id}):')
    for pais, count in top_paises.items():
        pct = count / len(hotel_df) * 100
        print(f'  - {pais}: {count:,} ({pct:.1f}%)')


In [ ]:
# Gráficos de tarta: Top países para los 3 hoteles con más reservas
top3_hoteles = reservas_hotel.head(3)['ID_HOTEL'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Colores distintivos (usando paleta corporativa y variantes)
colores_paises = [AUREA_COLORS['raw_sienna'], AUREA_COLORS['spicy_mustard'], AUREA_COLORS['luxor_gold'],
                  '#8B5A2B', '#C9A0DC', '#708090'] # Variantes y tonos neutros complementarios

for i, hotel_id in enumerate(top3_hoteles):
    hotel_df = df_reservas[df_reservas['ID_HOTEL'] == hotel_id]
    hotel_nombre = hotel_df['NOMBRE_HOTEL'].iloc[0]

    # Top 5 países + Otros
    paises = hotel_df['PAIS'].value_counts()
    top5 = paises.head(5)
    otros = len(hotel_df) - top5.sum()
    if otros > 0:
        top5 = pd.concat([top5, pd.Series({'Otros': otros})])

    axes[i].pie(top5.values, labels=top5.index, autopct='%1.1f%%', colors=colores_paises[:len(top5)], startangle=90)
    axes[i].set_title(f'{hotel_nombre}\n(ID: {hotel_id})')

plt.suptitle('Países de origen - Top 3 hoteles con más reservas', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Segmentos por hotel (top 8 hoteles)
print('Distribución de segmentos por hotel:')
print('=' * 80)

for hotel_id in reservas_hotel['ID_HOTEL'].head(8).tolist():
    hotel_df = df_reservas[df_reservas['ID_HOTEL'] == hotel_id]
    hotel_nombre = hotel_df['NOMBRE_HOTEL'].iloc[0]
    segmentos = hotel_df['SEGMENTO'].value_counts()

    print(f'\n{hotel_nombre} (ID: {hotel_id}):')
    for seg, count in segmentos.items():
        pct = count / len(hotel_df) * 100
        print(f'  - {seg}: {count:,} ({pct:.1f}%)')


In [ ]:
# Gráficos de tarta: Segmentos para los 3 hoteles con más reservas
fig, axes = plt.subplots(1, 3, figsize=(12, 5))

# Colores corporativos y complementarios elegantes
colores_segmentos = [AUREA_COLORS['spicy_mustard'], AUREA_COLORS['raw_sienna'], AUREA_COLORS['luxor_gold'],
                     '#5D4037', '#8D6E63', '#A1887F', '#D7CCC8', '#EFEBE9', '#607D8B', '#455A64']

# To collect all unique labels and create a single legend
all_labels = []
all_handles = []

for i, hotel_id in enumerate(top3_hoteles):
    hotel_df = df_reservas[df_reservas['ID_HOTEL'] == hotel_id]
    hotel_nombre = hotel_df['NOMBRE_HOTEL'].iloc[0]

    segmentos = hotel_df['SEGMENTO'].value_counts()

    # Solo mostrar porcentajes mayores a 5% en la tarta
    def autopct_func(pct):
        return f'{pct:.1f}%' if pct > 5 else ''

    wedges, texts, autotexts = axes[i].pie(segmentos.values, autopct=autopct_func,
                                           colors=[colores_segmentos[k % len(colores_segmentos)] for k in range(len(segmentos))], startangle=90,
                                           pctdistance=0.7)

    # Collect handles and labels for the global legend
    for j, segment_label in enumerate(segmentos.index):
        if segment_label not in all_labels:
            all_labels.append(segment_label)
            # Create a dummy handle for the legend based on color
            all_handles.append(plt.Line2D([0], [0], marker='o', color='w', label=segment_label,
                                          markerfacecolor=colores_segmentos[j % len(colores_segmentos)], markersize=8))


    # Leyenda fuera de la tarta
    # Remove individual legends
    # axes[i].legend(wedges, segmentos.index, loc='lower center', bbox_to_anchor=(0.5, -0.15),
    #                fontsize=8, ncol=2)
    axes[i].set_title(f'{hotel_nombre}\n(ID: {hotel_id})', fontsize=10)

plt.suptitle('Segmentos de mercado - Top 3 hoteles con más reservas', fontsize=12, fontweight='bold')

# Create a single legend for the entire figure
# Sort labels alphabetically for consistent legend order
sorted_legend_items = sorted(zip(all_labels, all_handles), key=lambda item: item[0])
sorted_labels = [item[0] for item in sorted_legend_items]
sorted_handles = [item[1] for item in sorted_legend_items]

fig.legend(sorted_handles, sorted_labels, loc='lower center', bbox_to_anchor=(0.5, 0.005),
           fancybox=True, shadow=True, ncol=5, fontsize=8)

plt.tight_layout() # Adjust layout to make space for the global legend
plt.subplots_adjust(bottom=0.015) # More space for legend
plt.show()

In [ ]:
# Fuentes de negocio por hotel (top 8)
print('Distribución de fuentes de negocio por hotel:')
print('=' * 80)

for hotel_id in reservas_hotel['ID_HOTEL'].head(8).tolist():
    hotel_df = df_reservas[df_reservas['ID_HOTEL'] == hotel_id]
    hotel_nombre = hotel_df['NOMBRE_HOTEL'].iloc[0]
    fuentes = hotel_df['FUENTE_NEGOCIO'].value_counts()

    print(f'\n{hotel_nombre} (ID: {hotel_id}):')
    for fuente, count in fuentes.items():
        pct = count / len(hotel_df) * 100
        print(f'  - {fuente}: {count:,} ({pct:.1f}%)')


In [ ]:
# Gráfico de barras agrupadas: Fuente de negocio para top 3 hoteles
fuentes_todas = df_reservas['FUENTE_NEGOCIO'].unique()

fig, ax = plt.subplots(figsize=(14, 6))

# Re-define top3_hoteles as it was not defined in the current scope
top3_hoteles = reservas_hotel.head(3)['ID_HOTEL'].tolist()

# Calcular porcentajes por hotel
datos_plot = []
nombres_hoteles = []
for hotel_id in top3_hoteles:
    hotel_df = df_reservas[df_reservas['ID_HOTEL'] == hotel_id]
    hotel_nombre = hotel_df['NOMBRE_HOTEL'].iloc[0]
    total = len(hotel_df)
    pcts = [(hotel_df['FUENTE_NEGOCIO'] == f).sum() / total * 100 for f in fuentes_todas[::-1]]
    datos_plot.append(pcts)
    nombres_hoteles.append(f'{hotel_nombre} (ID: {hotel_id})')

y = np.arange(len(fuentes_todas))
ancho = 0.25
colores_barras = [AUREA_COLORS['raw_sienna'], AUREA_COLORS['luxor_gold'], AUREA_COLORS['spicy_mustard']]

for i, hotel_id in enumerate(top3_hoteles):
    ax.barh(y + i*ancho, datos_plot[i], ancho, label=nombres_hoteles[i], color=colores_barras[i])

ax.set_xlabel('Porcentaje (%)', fontsize=12)
ax.set_title('Fuente de Negocio - Top 3 hoteles con más reservas', fontsize=14)
ax.set_yticks(y + ancho)
ax.set_yticklabels(fuentes_todas, fontsize=16)
ax.legend(loc='lower right', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Comparar medias de variables numéricas por hotel
variables_numericas = ['NOCHES', 'PAX', 'ADULTOS', 'VALHAB', 'VALPEN']

stats_hotel = df_reservas.groupby(['ID_HOTEL', 'NOMBRE_HOTEL'])[variables_numericas].mean().round(2)
stats_hotel = stats_hotel.reset_index()
stats_hotel = stats_hotel.merge(reservas_hotel[['ID_HOTEL', 'NUM_RESERVAS']], on='ID_HOTEL')
stats_hotel = stats_hotel.sort_values('NUM_RESERVAS', ascending=False)
stats_hotel['ETIQUETA'] = stats_hotel.apply(lambda x: f"{x['NOMBRE_HOTEL']} (ID: {x['ID_HOTEL']})", axis=1)

print('Estadísticas medias por hotel:')
print('=' * 110)
print(stats_hotel[['ID_HOTEL', 'NOMBRE_HOTEL', 'NOCHES', 'PAX', 'ADULTOS', 'VALHAB', 'VALPEN']].to_string(index=False))


In [ ]:
# Barras horizontales: Media de noches y PAX por hotel (top 8)
top8_hoteles = reservas_hotel.head(8)['ID_HOTEL'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Datos para top 8
stats_top8 = stats_hotel[stats_hotel['ID_HOTEL'].isin(top8_hoteles)].sort_values('NOCHES', ascending=True)

# Noches
axes[0].barh(stats_top8['ETIQUETA'], stats_top8['NOCHES'], color=AUREA_COLORS['raw_sienna'])
for i, (idx, row) in enumerate(stats_top8.iterrows()):
    axes[0].text(row['NOCHES'] + 0.1, i, f"{row['NOCHES']:.1f}", va='center', fontsize=14)
axes[0].set_xlabel('Media de noches')
axes[0].tick_params(axis='y', labelsize=14)
axes[0].set_title('Media de noches de estancia\n(Top 8 hoteles por volumen)')

# PAX
stats_top8_pax = stats_hotel[stats_hotel['ID_HOTEL'].isin(top8_hoteles)].sort_values('PAX', ascending=True)
axes[1].barh(stats_top8_pax['ETIQUETA'], stats_top8_pax['PAX'], color=AUREA_COLORS['luxor_gold'])
for i, (idx, row) in enumerate(stats_top8_pax.iterrows()):
    axes[1].text(row['PAX'] + 0.05, i, f"{row['PAX']:.1f}", va='center', fontsize=9)
axes[1].set_xlabel('Media de huéspedes')
axes[1].set_title('Media de huéspedes (PAX)\n(Top 8 hoteles por volumen)')

plt.tight_layout()
plt.show()


In [ ]:
# Crear tabla resumen con criterios de selección
resumen = cancelaciones_hotel.merge(paises_por_hotel, on=['ID_HOTEL', 'NOMBRE_HOTEL'])
resumen = resumen.sort_values('TOTAL_RESERVAS', ascending=False)

print('RESUMEN DE HOTELES - CRITERIOS PARA SELECCIÓN DEL MODELO')
print('=' * 110)
print(f'{"ID":<6} {"HOTEL":<35} {"RESERVAS":>12} {"CANCELADOS":>12} {"TASA %":>10} {"PAÍSES":>10}')
print('-' * 110)
for idx, row in resumen.iterrows():
    print(f"{row['ID_HOTEL']:<6} {row['NOMBRE_HOTEL'][:33]:<35} {row['TOTAL_RESERVAS']:>12,} {row['CANCELADOS']:>12,} {row['TASA_CANCELACION']:>10} {row['NUM_PAISES']:>10}")


In [ ]:
# Identificar los 3 hoteles con más reservas
top3 = resumen.head(3)

total_reservas_top3 = top3['TOTAL_RESERVAS'].sum()
pct_total = total_reservas_top3 / df_reservas.shape[0] * 100

print('\n' + '=' * 80)
print('TOP 3 HOTELES POR VOLUMEN DE RESERVAS:')
print('=' * 80)

for idx, row in top3.iterrows():
    print(f"\n- ID {row['ID_HOTEL']}: {row['NOMBRE_HOTEL']}")
    print(f"  Reservas: {row['TOTAL_RESERVAS']:,}")
    print(f"  Tasa cancelación: {row['TASA_CANCELACION']}%")
    print(f"  Países de origen: {row['NUM_PAISES']}")

print(f'\n\nEstos 3 hoteles representan el {pct_total:.1f}% del total de reservas del dataset.')
print('=' * 80)


### 8.1. Conclusión y Selección de Hoteles

Del analisis realizado se confirma que la seleccion de los **3 hoteles con mayor volumen** (IDs 96, 106, 107) es la estrategia optima.

**Justificación:**
1. **Volumen de datos:** Concentran la mayor parte de las reservas, lo que garantiza suficientes datos para entrenar un modelo fiable.
2. **Representatividad de Cancelacion:** Presentan tasas de cancelación alineadas con el negocio, permitiendo que el modelo aprenda patrones reales.
3. **Diversidad:** Tienen una amplia variedad de países de origen y segmentos de cliente, lo que aporta robustez y generalidad al modelo.
4. **Impacto:** Al cubrir la mayoría de las reservas, las mejoras predicadas por el modelo tendrán un mayor impacto práctico en la operativa.

**Decision:**
Se procedera a filtrar el dataset para trabajar exclusivamente con estos 3 hoteles en las siguientes fases de modelado.


---
## 9\. Filtrado del Dataset

Basado en el analisis anterior, filtramos el dataset para quedarnos unicamente con las reservas de los hoteles seleccionados: **Complejo Costa Mujeres (96)**, **Complejo Riviera Maya (106)** y **Complejo Punta Cana (107)**.

Esto nos permite focalizar el analisis y el modelado en una muestra representativa y robusta.


In [ ]:
# Definimos los IDs de los hoteles seleccionados
HOTELES_SELECCIONADOS = [96, 106, 107]

print(f'Dimensiones del dataset original: {df_reservas.shape}')

# Filtramos el DataFrame
df_filtered = df_reservas[df_reservas['ID_HOTEL'].isin(HOTELES_SELECCIONADOS)].copy()

print(f'Dimensiones del dataset filtrado: {df_filtered.shape}')
print(f'Reducción de datos: {100 - (len(df_filtered)/len(df_reservas)*100):.2f}%')

# Verificamos que solo queden los hoteles seleccionados
print('\nHoteles en el dataset filtrado:')
print(df_filtered[['ID_HOTEL', 'NOMBRE_HOTEL']].drop_duplicates().sort_values('ID_HOTEL'))


In [ ]:
# Reasignamos df_reservas al dataset filtrado para continuar el analisis con este subset
df_reservas = df_filtered

# Eliminamos df_filtered para liberar memoria
del df_filtered

print('[OK] Dataset filtrado y asignado a df_reservas. Listo para siguientes pasos.')


---
## 10\. Verificacion de Calidad de Datos (Post-Filtrado)

Antes de proceder con el analisis univariante, verificamos la integridad de los datos criticos, especialmente el mapeo de tipos de habitacion.


In [ ]:
# Estadisticas iniciales
df_reservas.describe().T


In [ ]:
# Verificamos si hay tipos de habitacion sin mapear en el catalogo
tipos_sin_mapeo = df_reservas[df_reservas['NOMBRE_HABITACION'].isnull()]['TIPO_NORMALIZADO'].value_counts()

print(f"Hay {len(tipos_sin_mapeo)} tipos de habitación que no tienen traducción en el diccionario para los hoteles seleccionados:")
if len(tipos_sin_mapeo) > 0:
    print(tipos_sin_mapeo)
else:
    print("[OK] Todos los tipos de habitación están correctamente mapeados.")


In [ ]:
# -------------------------------------------------------------------------
# Analisis de Nulos: CLIENTE
# -------------------------------------------------------------------------

# Creamos un dataset filtrando solo los nulos de CLIENTE
nulls_cliente = df_reservas[df_reservas['CLIENTE'].isnull()]

if len(nulls_cliente) > 0:
    print(f"Se encontraron {len(nulls_cliente)} registros con CLIENTE nulo.")
    # Mostramos los nulos con columnas de contexto
    print(nulls_cliente[['NOMBRE_HOTEL', 'NOCHES', 'STATUS', 'PAX', 'COMERCIALIZADORA', 'VALPEN', 'VALHAB', 'GRATIS']])
else:
    print("[OK] No hay nulos en CLIENTE para los hoteles seleccionados.")


In [ ]:
# -------------------------------------------------------------------------
# Verificacion: NOMBRE_HOTEL
# -------------------------------------------------------------------------

# Verificamos si existen nulos en NOMBRE_HOTEL dentro del subset filtrado
nulls_hoteles = df_reservas[df_reservas['NOMBRE_HOTEL'].isnull()]

if len(nulls_hoteles) > 0:
    print(f"[ATENCION] Se encontraron {len(nulls_hoteles)} registros con NOMBRE_HOTEL nulo.")
    print(nulls_hoteles[['ID_HOTEL', 'NOMBRE_HOTEL', 'STATUS']])
    print("\nIDs afectados:")
    print(nulls_hoteles['ID_HOTEL'].value_counts())
else:
    print("[OK] Integridad Correcta: No hay nulos en NOMBRE_HOTEL para el subset seleccionado (IDs 96, 106, 107).")


---
## 11\. Limpieza y Tratamiento de Outliers (Sobre Subset Filtrado)

En esta seccion procedemos a limpiar inconsistencias y eliminar outliers en variables clave (`NOCHES`, fechas y consistencia de `PAX`).


In [ ]:
# ---------------------------------------------------------
# 1. Revisamos las 'NOCHES' = 0
# ---------------------------------------------------------

noches_cero = df_reservas[df_reservas['NOCHES'] == 0]
print("Reservas con 0 Noches por Hotel:")
print(noches_cero.value_counts('NOMBRE_HOTEL'))

# Filtramos el dataset para quitarnos las noches 0
df_limpio = df_reservas[df_reservas['NOCHES'] > 0].copy()
print(f"\nRegistros tras eliminar NOCHES=0: {len(df_limpio)}")


In [ ]:
# ---------------------------------------------------------
# 2. Revisamos Outliers en Noches (Corte en 30)
# ---------------------------------------------------------

def visualizar_y_revisar_outliers(df):
    # Filtramos solo el segmento de la hipótesis
    confirmadas = df[df['STATUS'].isin([1,2,3,4,9])]

    # 1. Calculamos el impacto antes de mostrar
    total_confirmadas = len(confirmadas)
    outliers_count = len(confirmadas[confirmadas['NOCHES'] > 30])
    porcentaje = (outliers_count / total_confirmadas) * 100

    print(f"📊 ANÁLISIS DE OUTLIERS (> 30 NOCHES):")
    print(f"De {total_confirmadas} reservas confirmadas, {outliers_count} superan las 30 NOCHES ({porcentaje:.2f}%).")
    print("-" * 50)

    # 2. Generamos el Display
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # Histograma
    sns.histplot(confirmadas['NOCHES'], bins=50, kde=True, color='#FFCC00', ax=axes[0])
    axes[0].axvline(30, color='red', linestyle='--', label='Límite Hipótesis (30 n)')
    axes[0].set_title('Distribución de NOCHES')
    axes[0].legend()

    # Boxplot para ver la "cola" de la distribución
    sns.boxplot(x=confirmadas['NOCHES'], color='#003366', ax=axes[1])
    axes[1].set_title('Boxplot: Identificación de Valores Extremos')

    # IMPORTANTE: Esto es lo que hace que se vea en Colab
    plt.tight_layout()
    plt.show()

visualizar_y_revisar_outliers(df_limpio)


In [ ]:
# Definimos el Umbral de noches
umbral_noches = 30

# Contabilizamos antes de limpiar el dataset
total_inicial = len(df_limpio)
reservas_fuera = len(df_limpio[(df_limpio['STATUS'].isin([1,2,3,4,9])) & (df_limpio['NOCHES'] > umbral_noches) ])

# Limpiamos el dataset
df_limpio = df_limpio[~((df_limpio['STATUS'].isin([1,2,3,4,9])) & (df_limpio['NOCHES'] > umbral_noches))].copy()

print(f"Hito: Eliminación de Outliers en Estancias (Segmento Vacacional)")
print(f"Registros eliminados: {reservas_fuera}")
print(f"Impacto relativo: {(reservas_fuera/total_inicial)*100:.4f}% del dataset.")
print(f"Estado del Dataset: LIMPIO Y LISTO.")
print(f"\nRegistros ahora: {len(df_limpio)}")

In [ ]:
# ---------------------------------------------------------
# 3. Verificacion de fechas futuras incorrectas
# ---------------------------------------------------------

# Confirmamos cuantas reservas tienen como fecha salida posterior a febrero 2024
fecha_salida_futura = df_limpio[df_limpio['SALIDA'] >= '2024-02-01']
print(f"\n{len(fecha_salida_futura)} Salidas posteriores a 2024-02-01:")
print(fecha_salida_futura[['NOMBRE_HOTEL', 'NOCHES', 'STATUS']].head())

# Verificamos todas las modificaciones posteriores al 2023
fecha_mod_futura = df_limpio[df_limpio['FECHA_MOD'] >= '2024-01-01']
print(f"\n{len(fecha_mod_futura)} Modificaciones posteriores a 2024-01-01:")
print(fecha_mod_futura[['NOMBRE_HOTEL', 'NOCHES', 'STATUS']].head())

# Filtramos para quitarnos las modificaciones futuras errorneas
# Asumimos que datos > 2024-01-01 en MODIFICACION son errores si estamos trabajando con 22/23
# Si SALIDA es real > 2024, podria ser valido, pero MODIFICACION suele ser el log del sistema

# df_limpio = df_limpio[df_limpio['FECHA_MOD'] < '2024-01-01'].copy()
df_limpio = df_limpio[~(df_limpio['FECHA_MOD'] >= '2024-01-01')].copy()
print(f"\nRegistros tras eliminar FECHA_MOD post 2024: {len(df_limpio)}")

In [ ]:
print('Nulos en FECHA_MOD: ', df_limpio['FECHA_MOD'].isna().sum())
filtro_FECHA_MOD_nulo = df_limpio['FECHA_MOD'].isna()
# comprobamos FECHA_MOD nulo, imputamos con 1 día posterior a la fecha LLEGADA (porque hay muchas así)
# Aseguramos que la columna LLEGADA sea datetime para estas filas antes de sumar
df_limpio.loc[filtro_FECHA_MOD_nulo, 'FECHA_MOD'] = pd.to_datetime(df_limpio.loc[filtro_FECHA_MOD_nulo, 'LLEGADA'], errors='coerce') + pd.Timedelta(days=1)
df_limpio[filtro_FECHA_MOD_nulo][['FECHA_MOD','LLEGADA']]
print('Nulos ahora en FECHA_MOD: ', df_limpio['FECHA_MOD'].isna().sum())
print(f"\nRegistros ahora: {len(df_limpio)}")

In [ ]:
# ---------------------------------------------------------
# 4. Analisis de Inconsistencias: NOCHES vs FECHAS
# ---------------------------------------------------------

# Modificamos df_limpio para asegurarnos que NOCHES coincida con fechas

# 2. Calculamos la diferencia real en días
dias_calculados = (df_limpio['SALIDA'] - df_limpio['LLEGADA']).dt.days

# 3. Filtramos las filas donde NO coinciden (Mismatches)
errores_noches = df_limpio[df_limpio['NOCHES'] != dias_calculados].copy()

# 4. Resultados
if errores_noches.empty:
    print("✅ Todo correcto: La columna NOCHES coincide perfectamente con (SALIDA - LLEGADA).")
else:
    print(f"⚠️ Se encontraron {len(errores_noches)} discrepancias en NOCHES.")
    # Agregamos la columna de lo que "debería ser" para comparar visualmente
    errores_noches['CALCULO_REAL'] = (errores_noches['SALIDA'] - errores_noches['LLEGADA']).dt.days
    # Mostramos las columnas relevantes
    print(errores_noches[['LLEGADA', 'SALIDA', 'NOCHES', 'CALCULO_REAL']].head(10))


In [ ]:
# ---------------------------------------------------------
# 5. Analisis de Inconsistencias: PAX vs DETALLE PERSONAS
# ---------------------------------------------------------

# 1. Calculamos la suma real
suma_personas = (df_limpio['ADULTOS'].fillna(0) + df_limpio['NENES'].fillna(0))

# 2. Filtramos donde NO coinciden
errores_pax = df_limpio[df_limpio['PAX'] != suma_personas].copy()

# 3. Resultados
if errores_pax.empty:
    print("✅ Todo correcto: La columna PAX es exactamente la suma de ADULTOS + NENES.")
else:
    print(f"⚠️ Se encontraron {len(errores_pax)} discrepancias en PAX.")

    # Agregamos la columna de cálculo para ver la diferencia
    errores_pax['SUMA_REAL'] = (errores_pax['ADULTOS'].fillna(0) + errores_pax['NENES'].fillna(0))

    # Mostramos las primeras filas con error
    print(errores_pax[['ADULTOS', 'NENES', 'PAX', 'SUMA_REAL']].head(10))


In [ ]:
# ---------------------------------------------------------
# 6. Limpieza final de inconsistencias
# ---------------------------------------------------------

# Dropeamos las incosistencias de PAX y NOCHES

# 1. Definimos la condición de error en NOCHES
diferencia_dias = (df_limpio['SALIDA'] - df_limpio['LLEGADA']).dt.days
mask_error_noches = df_limpio['NOCHES'] != diferencia_dias

# 2. Definimos la condición de error en PAX
suma_personas = df_limpio['ADULTOS'].fillna(0) + df_limpio['NENES'].fillna(0)
mask_error_pax = df_limpio['PAX'] != suma_personas

# 3. Combinamos ambas condiciones (OR logic: si falla noches O falla pax)
filas_a_eliminar = df_limpio[mask_error_noches | mask_error_pax].index

# Informamos antes de borrar
print(f"Se eliminarán {len(filas_a_eliminar)} filas inconsistentes en total.")

# 4. Ejecutamos el DROP
df_limpio = df_limpio.drop(filas_a_eliminar)

# 5. Reasignamos df_reservas al dataset limpio
df_reservas = df_limpio
del df_limpio

# 6. Confirmación final
print("✅ Filas eliminadas correctamente.")
print(f"Nuevo tamaño final del dataset df_reservas: {df_reservas.shape}")


quitamos filas que no tienen REGIMEN All Inclusive

In [ ]:
# ---------------------------------------------------------
# 7. Limpieza filas que no tienen REGIMEN All Inclusive
# ---------------------------------------------------------

display(df_reservas[df_reservas['REGIMEN']!='All Inclusive'].groupby('REGIMEN')['ID_RESERVA'].count().reset_index())
display(df_reservas[df_reservas['REGIMEN']!='All Inclusive'].shape[0])

tamaño_antes = df_reservas.shape[0]
df_reservas = df_reservas[~(df_reservas['REGIMEN']!='All Inclusive')]

tamaño_despues = df_reservas.shape[0]
quitados = tamaño_antes - tamaño_despues
print(f"quitados {quitados}, tamaño_despues {tamaño_despues}")

In [ ]:
# -------------------------------------------------------------------------
# Analisis de Nulos: REGIMEN
# -------------------------------------------------------------------------

# Creamos un dataset filtrando solo los nulos de REGIMEN
nulls_regimen = df_reservas[df_reservas['REGIMEN'].isnull()]

if len(nulls_regimen) > 0:
    print(f"Se encontraron {len(nulls_regimen)} registros con REGIMEN nulo.")
    # Mostramos los nulos con columnas de contexto
    print(nulls_regimen[['NOMBRE_HOTEL', 'NOCHES', 'STATUS', 'PAX', 'COMERCIALIZADORA', 'VALPEN', 'VALHAB', 'GRATIS']])

    # Distribucion por hotel
    print("\nDistribución por Hotel:")
    print(nulls_regimen['NOMBRE_HOTEL'].value_counts(normalize=True)*100)
else:
    print("[OK] No hay nulos en REGIMEN para los hoteles seleccionados.")


In [ ]:
# Estadisticas iniciales
df_reservas.describe().T


---
## 12\. Feature Engineering


### 12.1. CANCELADO - Definición de la Variable Target
Eliminamos los registros con `STATUS = 9` (No Show / Especiales) para centrarnos en reservas confirmadas vs canceladas, y creamos la variable binaria `CANCELADO`.


In [ ]:
# 1. Filtrado de STATUS 9 (No Show / Especial)
count_status_9 = len(df_reservas[df_reservas['STATUS'] == 9])
print(f"Registros con STATUS=9 eliminados: {count_status_9}")

df_reservas = df_reservas[df_reservas['STATUS'] != 9].copy()

# 2. Creacion de variable target binaria
# 1=OK, 2=? (Asumimos OK o Pendiente), 3=Cancelada, 4=? (Asumimos Cancelada)
# Segun logica usuario: Confirmado=[1,2], Cancelado=[3,4,5]
confirmado = [1, 2]
anulado = [3, 4, 5]

df_reservas['CANCELADO'] = np.where(df_reservas['STATUS'].isin(confirmado), 0, 1)

print("\nDistribución del Target (CANCELADO):")
print(df_reservas['CANCELADO'].value_counts())
print("\nDistribución Porcentual:")
print(df_reservas['CANCELADO'].value_counts(normalize=True))

print(f"\ntamaño ahora {df_reservas.shape[0]}")

In [ ]:
# Para calcular el LEAD_TIME_CANCELATION sustraemos la fecha de FECHA_TOMA - FECHA_CANCELACION
# Nota: FECHA_CANCELACION < FECHA_TOMA seria un error logico (cancelar antes de reservar)

df_reservas['LEAD_TIME_CANCELATION'] = (df_reservas['FECHA_CANCELACION'].dt.normalize() - df_reservas['FECHA_TOMA'].dt.normalize()).dt.days

# 2 Verificación de seguridad
print("--- Validación de la columna LEAD_TIME_CANCELATION ---")
print(df_reservas[['FECHA_CANCELACION', 'FECHA_TOMA', 'LEAD_TIME_CANCELATION']].head(10))

# 1. Creamos el filtro para valores negativos en Lead Time de Cancelación
# (Recuerda que esto solo existe si target == 1)
mask_negativo_c = df_reservas['LEAD_TIME_CANCELATION'] < 0

# 2. Mostramos las columnas clave con los nombres exactos del DataFrame
lead_time_c_erroneos = df_reservas.loc[mask_negativo_c,
                                    ['ID_RESERVA', 'FECHA_CANCELACION', 'FECHA_TOMA', 'LEAD_TIME_CANCELATION']]

print(f"\nNúmero de reservas con 'Antelación de Cancelación' negativa: {len(lead_time_c_erroneos)}")

if len(lead_time_c_erroneos) > 0:
    display(lead_time_c_erroneos.sort_values(by='LEAD_TIME_CANCELATION').head(10))
    # Podriamos decidir eliminarlas o asumir error en fecha cancelacion
    # Por ahora solo diagnosticamos
else:
    print("¡Perfecto! No hay cancelaciones posteriores a la fecha de FECHA_TOMA (lógica temporal correcta).")

print(f"tamaño ahora {df_reservas.shape[0]}")

### 12.2. VALOR_RESERVA - Valor de Reserva y Normalización Monetaria

Calculamos el valor real de la reserva considerando si es comercializadora o no, y normalizamos todas las divisas a Dólares Americanos (USD) utilizando tipos de cambio históricos.


In [ ]:
# 1. Calculo de VALOR_RESERVA
print("Calculando VALOR_RESERVA unificado...")

# Definimos condiciones
condiciones_rev = [
    (df_reservas['COMERCIALIZADORA'] == 0),
    (df_reservas['COMERCIALIZADORA'] == 1)
]

# Definimos el calculo para cada condicion
logica_rev = [
    (df_reservas['VALHAB'] + df_reservas['VALPEN'] + df_reservas['VALSERV'] + df_reservas['VALFIJOS']), # Caso 0
    (df_reservas['CMVALHAB'] + df_reservas['CMVALPEN'] + df_reservas['CMCVALSERV']) # Caso 1
]

# Creamos columna VALOR_RESERVA
df_reservas['VALOR_RESERVA'] = np.select(condiciones_rev, logica_rev, default=0)

# 2. Verificación de seguridad
print("--- Validación de la columna VALOR_RESERVA ---")
print(df_reservas[['COMERCIALIZADORA', 'VALHAB', 'VALPEN', 'VALSERV', 'VALFIJOS', 'CMVALHAB', 'CMVALPEN', 'CMCVALSERV', 'VALOR_RESERVA']].head(10))

# 3. Verificamos cuantos valor reserva tenemos en negativos
mask_negativos = df_reservas['VALOR_RESERVA'] < 0
negativos_count = mask_negativos.sum()

if negativos_count > 0:
    print(f"\n⚠️ Detectados {negativos_count} registros con VALOR_RESERVA negativo.")
    print(df_reservas[mask_negativos]['COMERCIALIZADORA'].value_counts())

    # Filtramos el dataset para quitarnoslos
    df_reservas = df_reservas[df_reservas['VALOR_RESERVA'] >= 0 ].copy()
    print(f"✅ Filtro de negativos aplicado. Reservas actuales: {len(df_reservas)}")
else:
    print("\n✅ No hay valores negativos en VALOR_RESERVA.")


**Unificación y Normalización de Divisas (a USD)**

In [ ]:
# ==============================
# Mapeo de monedas
# ==============================

valor_antes = df_reservas.groupby(by=['ID_MONEDA', 'MONEDA']).agg(VALOR_RESERVA_media=('VALOR_RESERVA', 'mean'),VALOR_RESERVA_mediana=('VALOR_RESERVA', 'median'))

# 1. Extraemos los pares únicos de ID_MONEDA y MONEDA
mapeo_monedas = df_reservas[['ID_MONEDA', 'MONEDA']].drop_duplicates().sort_values(by='ID_MONEDA')

print("--- Diccionario de Monedas en Aurea (América) ---")
print(mapeo_monedas.to_string(index=False))

# ==============================
# Creacion df de exchange
# ==============================

# 1. Creamos el DataFrame de referencia para tipos de cambio
# IDs basados en el mapeo_monedas del dataset (ajustados a los valores comunes)
# Asumimos los IDs estandar: 700=USD, 1500=MXN, 900=DOP, 1400=JMD (o similar, segun el print anterior)
# Nota: El codigo a continuacion se adapta dinamicamente si el mapeo difiere,
# pero necesitamos los TICKERS de Yahoo Finance correctos.
data_referencia = {
    'ID_MONEDA': [700, 1500, 900, 1400, 500], # Añadido 500 (EUR) por seguridad si existe
    'MONEDA_REF': ['USD', 'MXN', 'DOP', 'JMD', 'EUR'],
    'TICKER': ['USDUSD=X', 'MXNUSD=X', 'DOPUSD=X', 'JMDUSD=X', 'EURUSD=X']
}
df_conversion = pd.DataFrame(data_referencia)

# Filtramos solo las monedas que realmente existen en nuestro dataset
monedas_presentes = df_reservas['ID_MONEDA'].unique()
df_conversion = df_conversion[df_conversion['ID_MONEDA'].isin(monedas_presentes)].copy()

print("\nMonedas a convertir:")
print(df_conversion)

# 2. Descarga de datos histórico (Yahoo Finance)
tickers = df_conversion['TICKER'].unique().tolist()
if len(tickers) > 0:
    print("\nDescargando tasas de cambio (2022-2023)...")
    data_api = yf.download(tickers, start="2022-01-01", end="2023-12-31")['Close']

    # 3. Calculamos medianas por año
    # Manejo de Series vs DataFrame si solo hay 1 ticker
    if isinstance(data_api, pd.Series):
         # Si es serie, renombrarla con el ticker unico
         data_api = data_api.to_frame(name=tickers[0])

    mediana_2022 = data_api[data_api.index.year == 2022].median()
    mediana_2023 = data_api[data_api.index.year == 2023].median()

    # 4. Mapeamos los valores al DataFrame de conversión
    # Nota: Yahoo da valor "Moneda -> USD" o "USD -> Moneda"?
    # Tickers como MXNUSD=X dan cuantos USD vale 1 MXN. Si usamos eso, multiplicamos.
    # Si usamos USDMXN=X, dividimos.
    # Yahoo 'MXNUSD=X' es Tasa: 1 MXN = X.XX USD. Correcto para multiplicar.

    df_conversion['MEDIANA_2022'] = df_conversion['TICKER'].map(mediana_2022).fillna(1.0)
    df_conversion['MEDIANA_2023'] = df_conversion['TICKER'].map(mediana_2023).fillna(1.0)

    # Caso especial USD (ID 700): Tasa es siempre 1
    df_conversion.loc[df_conversion['MONEDA_REF'] == 'USD', ['MEDIANA_2022', 'MEDIANA_2023']] = 1.0

    print("✅ Tabla de Conversión (a USD) creada con éxito:")
    print(df_conversion.to_string(index=False))

    # ==============================
    # Merge DF y Calculo final
    # ==============================

    # 1. Extraemos el año de LLEGADA
    df_reservas['ANIO_LLEGADA'] = df_reservas['LLEGADA'].dt.year

    # 2. Cruzamos con la tabla de conversión
    df_reservas = df_reservas.merge(df_conversion[['ID_MONEDA', 'MEDIANA_2022', 'MEDIANA_2023']],
                                 on='ID_MONEDA', how='left')

    # Rellenamos tasas nulas con 1 (asumimos USD o error menor)
    df_reservas['MEDIANA_2022'] = df_reservas['MEDIANA_2022'].fillna(1.0)
    df_reservas['MEDIANA_2023'] = df_reservas['MEDIANA_2023'].fillna(1.0)

    # 3. Creamos la columna VALOR_RESERVA_USD
    df_reservas['VALOR_RESERVA_USD'] = np.where(
        df_reservas['ANIO_LLEGADA'] == 2022,
        df_reservas['VALOR_RESERVA'] * df_reservas['MEDIANA_2022'],
        df_reservas['VALOR_RESERVA'] * df_reservas['MEDIANA_2023']
    )

    # 4. Limpieza final
    # Reemplazamos la variable original por la normalizada en USD
    df_reservas['VALOR_RESERVA'] = df_reservas['VALOR_RESERVA_USD']

    columnas_drop = ['MEDIANA_2022', 'MEDIANA_2023', 'ANIO_LLEGADA', 'VALOR_RESERVA_USD', 'LEAD_TIME_CANCELATION']
    df_reservas.drop(columns=columnas_drop, inplace=True, errors='ignore')

    print("\n¡Éxito! Valores normalizados a USD.")
else:
    print("⚠️ No se encontraron monedas para convertir o error en tickers.")


In [ ]:
display('VALOR_RESERVA antes',valor_antes)
print("")
display('VALOR_RESERVA ahora',
  df_reservas.groupby(by=['ID_MONEDA', 'MONEDA']).agg(VALOR_RESERVA_media=('VALOR_RESERVA', 'mean'),VALOR_RESERVA_mediana=('VALOR_RESERVA', 'median'))
)

### 12.3. ADR (Average Daily Rate)
ADR = VALOR_RESERVA / NOCHES


In [ ]:
def get_ADR(df_reservas):
  # Para calcular el ADR dividimos el VALOR_RESERVA / NOCHES
  # Nota: Ya nos aseguramos que NOCHES > 0 en la limpieza, asi que no habra division por cero.

  df_reservas['ADR'] = df_reservas['VALOR_RESERVA'] / df_reservas['NOCHES']

  # 2 Verificación de seguridad
  print("--- Validación de la columna ADR (USD) ---")
  print(df_reservas[['NOCHES', 'VALOR_RESERVA', 'ADR']].head(10))

  # Estadisticas rapidas del nuevo ADR
  print("\nEstadisticas de ADR:")
  print(df_reservas['ADR'].describe())

  return df_reservas

In [ ]:
df_reservas = get_ADR(df_reservas)

#### Outliers DE ADR

In [ ]:
df_reservas['ADR'].describe()

Quitar ADR=0

In [ ]:
df_reservas[df_reservas['ADR']==0].shape[0]

In [ ]:
df_reservas[df_reservas['ADR']==0][['ADR','GRATIS','CANCELADO']].value_counts()

In [ ]:
tamaño_antes = df_reservas.shape[0]
tamaño_antes

In [ ]:
# quitamos los ADR=0
df_reservas = df_reservas[~(df_reservas['ADR']==0)]

In [ ]:
tamaño_despues = df_reservas.shape[0]
quitados = tamaño_antes - tamaño_despues
print(f"quitados {quitados}, tamaño_despues {tamaño_despues}")

Quitar outliers de ADR > 2000

In [ ]:
print("Nº observaciones: ",df_reservas['ADR'].count())

num = 2000
total_adr_count = df_reservas['ADR'].count()
adrs_greater_than_num = df_reservas[df_reservas['ADR'] > num]['ADR'].count()
percentage = (adrs_greater_than_num / total_adr_count) * 100
print(f"Nº observaciones 'ADR'> {num}: {adrs_greater_than_num} ({percentage:.2f}%)")

df_dummy = pd.DataFrame({'PlotType': ['Boxplot1', 'Boxplot2']})
g = sns.FacetGrid(df_dummy, col='PlotType', sharex=False, sharey=False, height=6, aspect=1.5)

ax_boxplot1 = g.axes.flat[0]
sns.boxplot(y=df_reservas['ADR'].dropna(), ax=ax_boxplot1)
ax_boxplot1.set_title(f'ADR actual (Average Daily Rate)')
ax_boxplot1.set_ylabel('ADR')
ax_boxplot1.grid(True)

ax_boxplot2 = g.axes.flat[1]
sns.boxplot(y=df_reservas[df_reservas['ADR']<num]['ADR'].dropna(), ax=ax_boxplot2)
ax_boxplot2.set_title(f'ADR < {num} (Average Daily Rate)')
ax_boxplot2.set_ylabel('ADR')
ax_boxplot2.grid(True)

g.tight_layout()
plt.show()

In [ ]:
tamaño_antes = df_reservas.shape[0]
tamaño_antes

In [ ]:
# quitamos los ADR>2000
df_reservas = df_reservas[~(df_reservas['ADR']>2000)]

In [ ]:
tamaño_despues = df_reservas.shape[0]
quitados = tamaño_antes - tamaño_despues
print(f"quitados {quitados}, tamaño_despues {tamaño_despues}")

### 12.4. REV_PAX (Revenue per Pax)
REV_PAX = VALOR_RESERVA / PAX


In [ ]:
def get_REV_PAX(df_reservas):
  # Para calcular el REV_PAX dividimos el VALOR_RESERVA / PAX
  # Nota: PAX >= 1 garantizado por la limpieza anterior si hubiéramos filtrado PAX=0 (lo asumimos >0 en reservas normales)
  df_reservas['REV_PAX'] = df_reservas['VALOR_RESERVA'] / df_reservas['PAX']
  return df_reservas

In [ ]:
df_reservas = get_REV_PAX(df_reservas)

In [ ]:
# 2 Verificación de seguridad
print("--- Validación de la columna REV_PAX (USD) ---")
print(df_reservas[['PAX', 'VALOR_RESERVA', 'REV_PAX']].head(10))

# Estadisticas rapidas
print("\nEstadisticas de REV_PAX:")
print(df_reservas['REV_PAX'].describe())

### 12.5. LEAD_TIME - Cálculo y Limpieza

LEAD_TIME = LLEGADA - FECHA_TOMA

Calculamos los días de anticipación (`LEAD_TIME`) asegurándonos de normalizar las fechas (eliminando horas) para evitar errores de cálculo y valores negativos inconsistentes.


In [ ]:
def get_LEAD_TIME(df_reservas):
  tamaño_antes = df_reservas.shape[0]

  # Normalización y Cálculo
  # Normalizar (Quitar horas) para forzar cálculo por calendario puro
  df_reservas['FECHA_TOMA'] = df_reservas['FECHA_TOMA']#.dt.normalize()
  df_reservas['LLEGADA'] = df_reservas['LLEGADA']#.dt.normalize() # Por seguridad

  # Calcular Lead Time
  df_reservas['LEAD_TIME'] = (df_reservas['LLEGADA'].dt.normalize() - df_reservas['FECHA_TOMA'].dt.normalize()).dt.days

  return df_reservas

In [ ]:
df_reservas = get_LEAD_TIME(df_reservas)

In [ ]:
tamaño_antes = df_reservas.shape[0]

# Verificación de errores lógicos (0 días con fechas distintas)
filtro_error = (df_reservas['LLEGADA'] != df_reservas['FECHA_TOMA']) & (df_reservas['LEAD_TIME'] == 0)
print(f"Registros con error de cálculo tras normalizar: {df_reservas[filtro_error].shape[0]}")

# Filtrado de Negativos (Lead Times imposibles)
mask_negativo = df_reservas['LEAD_TIME'] < 0
lead_time_erroneos = df_reservas[mask_negativo]

if len(lead_time_erroneos) > 0:
    print(f"\nNúmero de reservas con Lead Time imposible (se eliminarán): {len(lead_time_erroneos)}")
    # print(lead_time_erroneos[['ID_RESERVA', 'FECHA_TOMA', 'LLEGADA', 'LEAD_TIME']].sort_values(by='LEAD_TIME').head(5))

    # Filtramos
    df_reservas = df_reservas[df_reservas['LEAD_TIME'] >= 0].copy()
    print("✅ Dataset saneado. Registros eliminados.")
else:
    print("✅ No hay Lead Times negativos.")

print(f"\nLead Time mínimo actual: {df_reservas['LEAD_TIME'].min()}")
print(f"\nLead Time máximo actual: {df_reservas['LEAD_TIME'].max()}")

tamaño_despues = df_reservas.shape[0]
quitados = tamaño_antes - tamaño_despues
print(f"quitados {quitados}, tamaño_despues {tamaño_despues}")

In [ ]:
# def get_LEAD_TIME(df_reservas):
#   tamaño_antes = df_reservas.shape[0]

#   # Normalización y Cálculo
#   # Normalizar (Quitar horas) para forzar cálculo por calendario puro
#   df_reservas['FECHA_TOMA'] = df_reservas['FECHA_TOMA']#.dt.normalize()
#   df_reservas['LLEGADA'] = df_reservas['LLEGADA']#.dt.normalize() # Por seguridad

#   # Calcular Lead Time
#   df_reservas['LEAD_TIME'] = (df_reservas['LLEGADA'].dt.normalize() - df_reservas['FECHA_TOMA'].dt.normalize()).dt.days

#   # Verificación de errores lógicos (0 días con fechas distintas)
#   filtro_error = (df_reservas['LLEGADA'] != df_reservas['FECHA_TOMA']) & (df_reservas['LEAD_TIME'] == 0)
#   print(f"Registros con error de cálculo tras normalizar: {df_reservas[filtro_error].shape[0]}")

#   # Filtrado de Negativos (Lead Times imposibles)
#   mask_negativo = df_reservas['LEAD_TIME'] < 0
#   lead_time_erroneos = df_reservas[mask_negativo]

#   if len(lead_time_erroneos) > 0:
#       print(f"\nNúmero de reservas con Lead Time imposible (se eliminarán): {len(lead_time_erroneos)}")
#       # print(lead_time_erroneos[['ID_RESERVA', 'FECHA_TOMA', 'LLEGADA', 'LEAD_TIME']].sort_values(by='LEAD_TIME').head(5))

#       # Filtramos
#       df_reservas = df_reservas[df_reservas['LEAD_TIME'] >= 0].copy()
#       print("✅ Dataset saneado. Registros eliminados.")
#   else:
#       print("✅ No hay Lead Times negativos.")

#   print(f"\nLead Time mínimo actual: {df_reservas['LEAD_TIME'].min()}")
#   print(f"\nLead Time máximo actual: {df_reservas['LEAD_TIME'].max()}")

#   tamaño_despues = df_reservas.shape[0]
#   quitados = tamaño_antes - tamaño_despues
#   print(f"quitados {quitados}, tamaño_despues {tamaño_despues}")
#   return df_reservas

### 12.6. LEAD_TIME_CANCELATION  

LEAD_TIME_CANCELATION = FECHA_TOMA - FECHA_CANCELACION

Calculamos la antelación con la que se canceló la reserva (días entre `FECHA_TOMA` y `FECHA_CANCELACION`). Esta variable solo tendrá valor para las reservas canceladas.


In [ ]:
print(f"tamaño ahora {df_reservas.shape[0]}")

---
## 13\. Análisis Univariante

Analizamos la distribución individual de las variables numéricas y categóricas más relevantes, así como la variable objetivo (`CANCELADO`).


### 13.1. Variable Target (CANCELADO)
Visualización de la proporción de cancelaciones global.


In [ ]:
# Datos
target_counts = df_reservas['CANCELADO'].value_counts()
labels = ['Confirmada (0)', 'Cancelada (1)']

# Grafico de Donut
plt.figure(figsize=(8, 8))
plt.pie(target_counts, labels=labels, autopct='%1.1f%%', startangle=90,
        colors=[COLOR_NO_CANCELADO, COLOR_CANCELADO],
        wedgeprops={'edgecolor': 'white', 'linewidth': 2}, textprops={'fontsize': 14})

# Circulo blanco central
centre_circle = plt.Circle((0,0),0.70,fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

plt.title('Distribución Global del Target (CANCELADO)', fontsize=16, fontweight='bold')
plt.show()

### 13.2. Variables Numéricas (Distribución y Outliers)
Visualizamos Histogramas y Boxplots para `ADR`, `VALOR_RESERVA`, `LEAD_TIME`, `NOCHES` y `REV_PAX`.


In [ ]:
# vars_nums = ['ADR', 'VALOR_RESERVA', 'LEAD_TIME', 'NOCHES', 'REV_PAX']
vars_nums = ['ADR', 'LEAD_TIME', 'NOCHES']

# Configuración Estética
sns.set_style("whitegrid")
plt.figure(figsize=(18, 5 * len(vars_nums)))

for i, var in enumerate(vars_nums):
    # Histograma (Distribución)
    plt.subplot(len(vars_nums), 2, 2*i + 1)
    sns.histplot(df_reservas[var], kde=True, color=PALETA_AUREA[0], bins=50)
    plt.title(f'Distribución de {var}', fontsize=12, fontweight='bold')
    plt.xlabel(var)
    plt.ylabel('Frecuencia')

    # Boxplot (Outliers)
    plt.subplot(len(vars_nums), 2, 2*i + 2)
    sns.boxplot(x=df_reservas[var], color=PALETA_AUREA[1])
    plt.title(f'Boxplot de {var}', fontsize=12, fontweight='bold')
    plt.xlabel(var)

plt.tight_layout()
plt.show()

### 13.3. Variables Categóricas (Rankings)
Analizamos las principales categorías de negocio: `'SEGMENTO'`, `'CLIENTE'`, `'NOMBRE_HABITACION'`, `'Categoria'`.


In [ ]:
df_reservas['CLIENTE'].value_counts()

In [ ]:
vars_cats = ['PAIS', 'NOMBRE_HABITACION', 'CLIENTE', 'SEGMENTO', 'FUENTE_NEGOCIO' ]

# plt.figure(figsize=(18, 6 * len(vars_cats)))

for i, var in enumerate(vars_cats):
    plt.figure(figsize=(12, 14))

    plt.subplot(len(vars_cats), 1, i + 1)

    data = df_reservas
    # Get the order by frequency for the current variable
    order = df_reservas[var].value_counts().index

    # If there are more than 15 unique values, filter `data` to include only the top 15
    if df_reservas[var].nunique() > 15:
        top_15_values = order[:15]
        data = df_reservas[df_reservas[var].isin(top_15_values)]
        order = top_15_values

    # Generate an extended palette to ensure explicit cycling for all categories
    num_colors_needed = len(order)
    # Calculate how many times the base palette needs to be repeated
    num_repeats = (num_colors_needed // len(PALETA_AUREA)) + 1
    # Create an extended palette by repeating PALETA_AUREA and slicing to the exact number of categories
    extended_palette = (PALETA_AUREA * num_repeats)[:num_colors_needed]

    # Grafico de barras
    ax = sns.countplot(y=var, data=data, order=order, palette=extended_palette, hue=var, legend=False)
    plt.title(f'Frecuencia por {var}', fontsize=14, fontweight='bold')
    plt.xlabel('Número de Reservas')
    plt.ylabel(var)

    # Etiquetas de datos
    for p in ax.patches:
        width = p.get_width()
        plt.text(width + 500, p.get_y() + p.get_height()/2,
                 '{:1.0f}'.format(width),
                 ha='left', va='center')

    plt.tight_layout()
    plt.show()

In [ ]:
df_reservas['NOMBRE_HABITACION'].value_counts()

In [ ]:
# NOMBRE_HABITACION (muchas categorias, grafico separado)
if 'NOMBRE_HABITACION' in df_reservas.columns:
    plt.figure(figsize=(14, 10))
    order = df_reservas['NOMBRE_HABITACION'].value_counts().index[:15]
    ax = sns.countplot(y='NOMBRE_HABITACION', data=df_reservas, order=order, palette='YlOrBr')
    plt.title('Frecuencia por NOMBRE_HABITACION (Top 15)', fontsize=14, fontweight='bold')
    plt.xlabel('Numero de Reservas')
    plt.ylabel('')
    for p in ax.patches:
        width = p.get_width()
        plt.text(width + 100, p.get_y() + p.get_height()/2, f'{int(width):,}', va='center')
    plt.tight_layout()
    plt.show()


---
## 14\. Analisis Bivariante

Cruces de variables contra el target CANCELADO.

### 14.1. Categoricas vs Target


In [ ]:
vars_cats_bi = ['PAIS', 'NOMBRE_HABITACION', 'CLIENTE', 'SEGMENTO', 'FUENTE_NEGOCIO', 'NOMBRE_HOTEL']

for var in vars_cats_bi:
    if var in df_reservas.columns:

        data = df_reservas.copy()

        top = ''

        # Si hay más de 15 categorías, filtramos el top 15
        if df_reservas[var].nunique() > 10:
          top_15_values = df_reservas[var].value_counts().head(10).index
          data = df_reservas[df_reservas[var].isin(top_15_values)].copy()
          top = 'TOP '

        ct = pd.crosstab(data[var], data['CANCELADO'], normalize='index') * 100
        ct = ct.sort_values(by=1, ascending=False)

        ax = ct.plot(kind='barh', stacked=True, figsize=(12, 6), color=['#D28B57', '#99782C'])
        plt.title(f'Tasa de Cancelacion por {top}{var}')
        plt.ylabel('Porcentaje (%)')
        plt.xlabel(var)
        plt.legend(['Confirmada', 'Cancelada'])
        plt.tight_layout()
        plt.show()

### 14.2. Numericas vs Target


In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_reservas, x='LEAD_TIME', hue='CANCELADO', fill=True, palette={0: COLOR_NO_CANCELADO, 1: COLOR_CANCELADO})
plt.title('Distribución de Densidad de LEAD_TIME por Estado de Reserva')
plt.xlabel('LEAD_TIME (Días)')
plt.ylabel('Densidad')
plt.legend(title='CANCELADO', labels=['Cancelado', 'No Cancelado'])
plt.tick_params(axis='y', left=False, labelleft=False) # Hide y-axis ticks and labels
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_reservas[df_reservas['LEAD_TIME']<600], x='LEAD_TIME', hue='CANCELADO', fill=True, palette={0: COLOR_NO_CANCELADO, 1: COLOR_CANCELADO})
plt.title('Distribución de Densidad de LEAD_TIME<600 por Estado de Reserva')
plt.xlabel('LEAD_TIME (Días)')
plt.ylabel('Densidad')
plt.legend(title='CANCELADO', labels=['Cancelado', 'No Cancelado'])
plt.tick_params(axis='y', left=False, labelleft=False) # Hide y-axis ticks and labels
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_reservas, x='ADR', hue='CANCELADO', fill=True, palette={0: COLOR_NO_CANCELADO, 1: COLOR_CANCELADO})
plt.title('Distribución de Densidad de ADR por Estado de Reserva')
plt.xlabel('ADR ($)')
plt.ylabel('Densidad')
plt.legend(title='CANCELADO', labels=['Cancelado', 'No Cancelado'])
plt.tick_params(axis='y', left=False, labelleft=False) # Hide y-axis ticks and labels
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_reservas, x='NOCHES', hue='CANCELADO', fill=True, palette={0: COLOR_NO_CANCELADO, 1: COLOR_CANCELADO})
plt.title('Distribución de Densidad de NOCHES por Estado de Reserva')
plt.xlabel('NOCHES')
plt.ylabel('Densidad')
plt.legend(title='CANCELADO', labels=['Cancelado', 'No Cancelado'])
plt.tick_params(axis='y', left=False, labelleft=False) # Hide y-axis ticks and labels
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_reservas, x='VALOR_RESERVA', hue='CANCELADO', fill=True, palette={0: COLOR_NO_CANCELADO, 1: COLOR_CANCELADO})
plt.title('Distribución de Densidad de VALOR_RESERVA por Estado de Reserva')
plt.xlabel('VALOR_RESERVA ($)')
plt.ylabel('Densidad')
plt.legend(title='CANCELADO', labels=['Cancelado', 'No Cancelado'])
plt.tick_params(axis='y', left=False, labelleft=False) # Hide y-axis ticks and labels
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_reservas, x='PAX', hue='CANCELADO', fill=True, palette={0: COLOR_NO_CANCELADO, 1: COLOR_CANCELADO})
plt.title('Distribución de Densidad de PAX por Estado de Reserva')
plt.xlabel('PAX (personas reserva)')
plt.ylabel('Densidad')
plt.legend(title='CANCELADO', labels=['Cancelado', 'No Cancelado'])
plt.tick_params(axis='y', left=False, labelleft=False) # Hide y-axis ticks and labels
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
vars_nums_bi = ['ADR', 'LEAD_TIME', 'NOCHES', 'VALOR_RESERVA']

for var in vars_nums_bi:
    if var in df_reservas.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x='CANCELADO', y=var, data=df_reservas, palette=['#D28B57', '#99782C'])
        plt.title(f'{var} vs Estado de Reserva')
        plt.xlabel('Estado (0=Confirmada, 1=Cancelada)')
        plt.tight_layout()
        plt.show()


### 14.3. Matriz de Correlacion


In [ ]:
cols_corr = ['NOCHES', 'PAX', 'ADULTOS', 'NENES', 'BEBES', 'SUPLETORIA', 'CUNAS',
       'VALOR_RESERVA', 'ADR', 'LEAD_TIME', 'REV_PAX', 'CANCELADO']
cols_corr = [c for c in cols_corr if c in df_reservas.columns]

plt.figure(figsize=(10, 8))
corr_matrix = df_reservas[cols_corr].corr()
sns.heatmap(corr_matrix, annot=True, cmap='YlOrBr', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlacion')
plt.tight_layout()
plt.show()


### 14.4. Estacionalidad


In [ ]:
df_reservas['MES_LLEGADA'] = df_reservas['LLEGADA'].dt.month
tasa_mes = df_reservas.groupby('MES_LLEGADA')['CANCELADO'].mean() * 100

plt.figure(figsize=(12, 6))
ax = sns.barplot(x=tasa_mes.index, y=tasa_mes.values, color='#6E550C')
plt.title('Tasa de Cancelacion por Mes de Llegada')
plt.xlabel('Mes')
plt.ylabel('Tasa de Cancelacion (%)')
plt.ylim(0, 60)

for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + 1, f'{height:.1f}%', ha='center')

plt.tight_layout()
plt.show()


## 15\. Feature Engineering Avanzado

### 15.1. HORA_TOMA
Hora de la Fecha de toma cuando se hace la reserva

Hacemos la transformación a Seno y Coseno

In [ ]:
def get_HORA_TOMA(df_reservas):
  df_reservas['HORA_TOMA'] = df_reservas['FECHA_TOMA'].dt.hour
  print('[OK] Columna HORA_TOMA creada correctamente.')
  display(df_reservas[['FECHA_TOMA', 'HORA_TOMA']].head())

  df_reservas['HORA_TOMA_SIN'] = np.sin(2 * np.pi * df_reservas['HORA_TOMA'] / 24)
  df_reservas['HORA_TOMA_COS'] = np.cos(2 * np.pi * df_reservas['HORA_TOMA'] / 24)

  return df_reservas

In [ ]:
df_reservas = get_HORA_TOMA(df_reservas)

In [ ]:
df_reservas['HORA_TOMA'].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax = sns.scatterplot(x=df_reservas['HORA_TOMA_SIN'], y=df_reservas['HORA_TOMA_COS'])
ax.set_title('HORA_TOMA cíclico')
ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
df_reservas[['HORA_TOMA_SIN', 'HORA_TOMA_COS']].round(2).value_counts()

### 15.2. MES_LLEGADA (MES_LLEGADA_SIN y MES_LLEGADA_COS)

In [ ]:
def get_MES_LLEGADA(df_reservas):
  df_reservas['LLEGADA'] = pd.to_datetime(df_reservas['LLEGADA'])
  df_reservas['MES_LLEGADA'] = df_reservas['LLEGADA'].dt.month

  df_reservas['MES_LLEGADA_SIN'] = np.sin(2 * np.pi * df_reservas['MES_LLEGADA'] / 12)
  df_reservas['MES_LLEGADA_COS'] = np.cos(2 * np.pi * df_reservas['MES_LLEGADA'] / 12)

  return df_reservas

In [ ]:
df_reservas = get_MES_LLEGADA(df_reservas)

In [ ]:
display(df_reservas[['LLEGADA', 'SALIDA', 'FECHA_TOMA', 'FECHA_MOD', 'MES_LLEGADA']])

print(np.sort(df_reservas['MES_LLEGADA'].unique()))

get_MES_LLEGADA(df_reservas)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax = sns.scatterplot(x=df_reservas['MES_LLEGADA_SIN'], y=df_reservas['MES_LLEGADA_COS'], s=50)
ax.set_title('MES_LLEGADA cíclico')
ax.grid(True)

plt.tight_layout()
plt.show()

### 15.3. TIPO_VIAJERO - Categorización por número de PAX
Creamos una nueva variable `TIPO_VIAJERO` para segmentar las reservas según el número total de ocupantes (`PAX`).


In [ ]:
def get_TIPO_VIAJERO(df_reservas):
  # 1. Definimos las condiciones basadas en el número de personas (PAX)
  condiciones_pax = [
      (df_reservas['PAX'] == 1),
      (df_reservas['PAX'] == 2),
      (df_reservas['PAX'] >= 3)
  ]

  # 2. Definimos las etiquetas correspondientes
  etiquetas_pax = ['Single', 'Parejas', 'Familias']

  # 3. Creamos la nueva columna 'TIPO_VIAJERO'
  df_reservas['TIPO_VIAJERO'] = np.select(condiciones_pax, etiquetas_pax, default='Otro')

  # Porcentaje
  print("\nDistribución Porcentual:")
  print(df_reservas['TIPO_VIAJERO'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

  print("Distribución de Tipo de Viajero:")
  print("-" * 30)
  resultado = df_reservas['TIPO_VIAJERO'].value_counts()
  print(resultado)

  return df_reservas


In [ ]:
df_reservas = get_TIPO_VIAJERO(df_reservas)

### 15.4. TIENE_FIDELIDAD - LOYALTY_REWARDS - Variables de Fidelidad (Aurea Rewards)
Analizamos la fidelidad de los clientes, creando indicadores booleanos para clientes fidelizados en general (`TIENE_FIDELIDAD`) y miembros del programa específico (`LOYALTY_REWARDS`).


In [ ]:
# Revisamos los ID_FIDELIDAD Y FIDELIDAD
mapeo_fidelidad = df_reservas[['ID_FIDELIDAD', 'FIDELIDAD']].drop_duplicates().sort_values(by='ID_FIDELIDAD')
print("--- RELACIÓN ID_FIDELIDAD vs CATEGORÍA ---")
print(mapeo_fidelidad.to_string(index=False))


In [ ]:
def get_LOYALTY_REWARDS_y_TIENE_FIDELIDAD(df_reservas):

  # 1. Creamos la columna TIENE_FIDELIDAD
  # Si es NaN ponemos 0 si tiene cualquier nombre ponemos 1
  df_reservas['TIENE_FIDELIDAD'] = np.where(df_reservas['FIDELIDAD'].isna(), 0, 1)

  # 2. Creamos la columna LOYALTY_REWARDS
  # Si FIDELIDAD es Aurea Rewards ponemos 1 si no ponemos 0
  df_reservas['LOYALTY_REWARDS'] = np.where(df_reservas['FIDELIDAD'] == 'Aurea Rewards', 1, 0)

  return df_reservas

In [ ]:
df_reservas = get_LOYALTY_REWARDS_y_TIENE_FIDELIDAD(df_reservas)

In [ ]:
# 3. Hacemos drop de la columna ID_FIDELIDAD (Redundante o poco util)
df_reservas.drop(columns=['ID_FIDELIDAD'], inplace=True)

# 4. Verificación rápida
print("\n--- RESUMEN DE NUEVAS VARIABLES (FIDELIDAD) ---")
print(f"Total Clientes Fidelizados: {df_reservas['TIENE_FIDELIDAD'].sum()}")
print(f"Total Aurea Rewards:    {df_reservas['LOYALTY_REWARDS'].sum()}")

# 5. Verificacion final de columnas disponibles
print(f"\nColumnas actuales en el dataset: {len(df_reservas.columns)}")
print(df_reservas.columns.tolist())

### 15.5. ES_GRUPO - Segmentación de Grupos (ID_MULTIPLE)
Identificamos las reservas que pertenecen a un grupo (`ID_MULTIPLE != 0`) creando la variable binaria `ES_GRUPO`, y eliminamos `ID_MULTIPLE`.


In [ ]:
def get_ES_GRUPO(df_reservas):
  # 1. Creamos la columna ES_GRUPO
  # Si ID_MULTIPLE es distinto a 0 entonces ES_GRUPO es 1 si no ponemos 0
  df_reservas['ES_GRUPO'] = np.where(df_reservas['ID_MULTIPLE'] != 0, 1, 0)

  # 2. Validación de Negocio
  total_grupos = df_reservas['ES_GRUPO'].sum()
  print(f"--- ANÁLISIS DE SEGMENTACIÓN DE GRUPOS ---")
  print(f"📦 Reservas de Grupo: {total_grupos} ({(total_grupos/len(df_reservas))*100:.2f}%)")
  print(f"👤 Reservas Individuales: {len(df_reservas) - total_grupos}")

  # 3. Borramos el ID_MULTIPLE para limpiar el dataset (ya tenemos su valor en la nueva columna)
  if 'ID_MULTIPLE' in df_reservas.columns:
      df_reservas.drop(columns=['ID_MULTIPLE'], inplace=True)
      print("\n✅ Variable 'ES_GRUPO' creada e 'ID_MULTIPLE' eliminado.")
  else:
      print("\n⚠️ La columna 'ID_MULTIPLE' no existe (quizas ya se borro).")

  return df_reservas

In [ ]:
df_reservas = get_ES_GRUPO(df_reservas)

### 15.6. PAIS_TOP_200

In [ ]:
def get_PAIS_TOP_200 (df_reservas):
  total_reservas_pais = df_reservas.groupby('PAIS')['ID_RESERVA'].count().reset_index()
  total_reservas_pais.columns = ['PAIS', 'Total_Reservas']

  # Top 200 paises
  top_200_paises = total_reservas_pais.sort_values(by='Total_Reservas', ascending=False).head(200)['PAIS'].tolist()

  # Nueva columna 'PAIS_TOP_200', con valor 'OTROS' para el resto
  df_reservas['PAIS_TOP_200'] = df_reservas['PAIS'].apply(lambda x: x if x in top_200_paises else 'OTROS')

  return df_reservas

In [ ]:
df_reservas = get_PAIS_TOP_200 (df_reservas)

In [ ]:
display(df_reservas['PAIS_TOP_200'].value_counts())

### 15.7. FUENTE_NEGOCIO_SEGMENTO_CLIENTE

In [ ]:
def get_FUENTE_NEGOCIO_SEGMENTO_CLIENTE (df_reservas):
  display('Valores de FUENTE_NEGOCIO:', df_reservas['FUENTE_NEGOCIO'].value_counts().size)
  display('Valores de SEGMENTO:', df_reservas['SEGMENTO'].value_counts().size)
  display('Valores de CLIENTE:', df_reservas['CLIENTE'].value_counts().size)
  df_reservas['FUENTE_NEGOCIO_SEGMENTO_CLIENTE'] = df_reservas['FUENTE_NEGOCIO'] + '_' + df_reservas['SEGMENTO'] + '_' + df_reservas['CLIENTE']
  display('Valores de FUENTE_NEGOCIO_SEGMENTO_CLIENTE:', df_reservas['FUENTE_NEGOCIO_SEGMENTO_CLIENTE'].value_counts().size)
  display(df_reservas['FUENTE_NEGOCIO_SEGMENTO_CLIENTE'].head(5))

  return df_reservas

In [ ]:
df_reservas = get_FUENTE_NEGOCIO_SEGMENTO_CLIENTE (df_reservas)

### 15.8. PREFIJO_HAB

#### Limpieza NOUSAR*

In [ ]:
# 707 NOMBRE_HABITACION comienzan con 'NOUSAR*'
display(df_reservas[df_reservas['NOMBRE_HABITACION'].str.startswith('NOUSAR*')]['NOMBRE_HABITACION'])
indice_nousar = df_reservas[df_reservas['NOMBRE_HABITACION'].str.startswith('NOUSAR*')].index

In [ ]:
# separar resto del nombre
display(df_reservas.loc[indice_nousar]['NOMBRE_HABITACION'].str.split('*')[:].str[1])
df_reservas.loc[indice_nousar,'NOMBRE_HABITACION'] = df_reservas.loc[indice_nousar,'NOMBRE_HABITACION'].str.split('*')[:].str[1]
display(df_reservas[df_reservas['NOMBRE_HABITACION'].str.startswith('NOUSAR*')]['NOMBRE_HABITACION'])

In [ ]:
# nombres a encontrar
df_reservas.loc[indice_nousar,'NOMBRE_HABITACION'].unique()

In [ ]:
# nombres del hotel CMU
filtro_cmu = df_reservas['NOMBRE_HABITACION'].str.startswith('CMU')
df_reservas.loc[filtro_cmu,'NOMBRE_HABITACION'].value_counts().index

In [ ]:
# filtamos junior suite que no son fs
filtro_junior_suite = df_reservas['NOMBRE_HABITACION'].str.contains('JUNIOR SUITE')
filtro_fs = df_reservas['NOMBRE_HABITACION'].str.contains('FS')
df_reservas[filtro_cmu & filtro_junior_suite & ~filtro_fs]['NOMBRE_HABITACION'].value_counts().index

In [ ]:
# primero corregimos las JS
filtro_js = df_reservas['NOMBRE_HABITACION'].str.contains('JS')
df_reservas[filtro_cmu & filtro_js]['NOMBRE_HABITACION'].value_counts().index

In [ ]:
filtro_cmu_js_bs = df_reservas['NOMBRE_HABITACION'] == 'CMU JS BS'
df_reservas.loc[filtro_cmu_js_bs, 'NOMBRE_HABITACION'] = 'CMU JUNIOR SUITE BS'

filtro_cmu_js_bs_ov = df_reservas['NOMBRE_HABITACION'] == 'CMU JS BS OV'
df_reservas.loc[filtro_cmu_js_bs_ov, 'NOMBRE_HABITACION'] = 'CMU JUNIOR SUITE BS OV'

filtro_cmu_js_bs_pov = df_reservas['NOMBRE_HABITACION'] == 'CMU JS BS POV'
df_reservas.loc[filtro_cmu_js_bs_pov, 'NOMBRE_HABITACION'] = 'CMU JUNIOR SUITE BS POV'

filtro_cmu_js_sw_bs = df_reservas['NOMBRE_HABITACION'] == 'CMU JS SW BS'
df_reservas.loc[filtro_cmu_js_sw_bs, 'NOMBRE_HABITACION'] = 'CMU JUNIOR SUITE SW BS'

filtro_cmu_amb_st_bs = df_reservas['NOMBRE_HABITACION'] == 'CMU AMB ST BS'
df_reservas.loc[filtro_cmu_amb_st_bs, 'NOMBRE_HABITACION'] = 'CMU AMBASSADOR ST BS'

In [ ]:
df_reservas.loc[indice_nousar,'NOMBRE_HABITACION'].value_counts()

#### Creación PREFIJO_HAB

In [ ]:
def get_PREFIJO_HAB(df_reservas):
  df_reservas['PREFIJO_HAB'] = [nombre.split(' ')[0] for nombre in df_reservas['NOMBRE_HABITACION']]
  filtro_cmu_fs = df_reservas['NOMBRE_HABITACION'].str.startswith('CMU FS')
  df_reservas.loc[filtro_cmu_fs, 'PREFIJO_HAB'] = 'CMU FS'
  df_reservas['PREFIJO_HAB'].unique()

  return df_reservas

In [ ]:
df_reservas = get_PREFIJO_HAB(df_reservas)

In [ ]:
df_reservas['PREFIJO_HAB'].value_counts()

### 15.9. NOMBRE_COMPLEJO

In [ ]:
def get_NOMBRE_COMPLEJO(df_reservas):

  condiciones = [
      df_reservas['NOMBRE_HOTEL'] == 'Complejo Riviera Maya',
      df_reservas['NOMBRE_HOTEL'] == 'Complejo Punta Cana',
      df_reservas['NOMBRE_HOTEL'] == 'Complejo Costa Mujeres'
  ]

  opciones = ['MAYA', 'CANA', 'MUJE']

  df_reservas['NOMBRE_COMPLEJO'] = np.select(condiciones, opciones, default='OTRO')

  display(df_reservas[['NOMBRE_COMPLEJO', 'NOMBRE_HOTEL']].value_counts())

  return df_reservas

In [ ]:
df_reservas = get_NOMBRE_COMPLEJO(df_reservas)

### 15.10 HOTEL_COMPLEJO

In [ ]:
def get_HOTEL_COMPLEJO(df_reservas):
  condiciones = [
      (df_reservas['NOMBRE_HOTEL'] == 'Complejo Riviera Maya') & (df_reservas['PREFIJO_HAB'].str.startswith('COL')),
      (df_reservas['NOMBRE_HOTEL'] == 'Complejo Riviera Maya') & (df_reservas['PREFIJO_HAB'].str.startswith('KAN')),
      (df_reservas['NOMBRE_HOTEL'] == 'Complejo Riviera Maya') & (df_reservas['PREFIJO_HAB'].str.startswith('SIG')),
      (df_reservas['NOMBRE_HOTEL'] == 'Complejo Riviera Maya') & (df_reservas['PREFIJO_HAB'].str.startswith('WS')),

      (df_reservas['NOMBRE_HOTEL'] == 'Complejo Punta Cana') & (df_reservas['PREFIJO_HAB'].str.startswith('BAV')),
      (df_reservas['NOMBRE_HOTEL'] == 'Complejo Punta Cana') & (df_reservas['PREFIJO_HAB'].str.startswith('PAL')),
      (df_reservas['NOMBRE_HOTEL'] == 'Complejo Punta Cana') & (df_reservas['PREFIJO_HAB'].str.startswith('PC')),
      (df_reservas['NOMBRE_HOTEL'] == 'Complejo Punta Cana') & (df_reservas['PREFIJO_HAB'].str.startswith('SIG')),

      (df_reservas['NOMBRE_HOTEL'] == 'Complejo Costa Mujeres') & (df_reservas['PREFIJO_HAB'].str.startswith('CMU FS')),
      (df_reservas['NOMBRE_HOTEL'] == 'Complejo Costa Mujeres') & (df_reservas['PREFIJO_HAB'].str.startswith('CMU')),
      (df_reservas['NOMBRE_HOTEL'] == 'Complejo Costa Mujeres') & (df_reservas['PREFIJO_HAB'].str.startswith('SIGC')),
      (df_reservas['NOMBRE_HOTEL'] == 'Complejo Costa Mujeres') & (df_reservas['PREFIJO_HAB'].str.startswith('SIG'))
  ]

  opciones = [
      'MAYA_COL', 'MAYA_KAN', 'MAYA_SIG', 'MAYA_WS',
      'CANA_BAV', 'CANA_PAL', 'CANA_PC', 'CANA_SIG',
      'MUJE_CMU_FS', 'MUJE_CMU', 'MUJE_SIGC', 'MUJE_SIG'
  ]

  df_reservas['HOTEL_COMPLEJO'] = np.select(condiciones, opciones, default='OTRO')

  display(df_reservas[['NOMBRE_HOTEL','PREFIJO_HAB','HOTEL_COMPLEJO']].value_counts().sort_index())

  return df_reservas

In [ ]:
df_reservas = get_HOTEL_COMPLEJO(df_reservas)

### 15.11. COMPLEJO_RESERVA = HOTEL_COMPLEJO + NOMBRE_HABITACION

In [ ]:
def get_COMPLEJO_RESERVA(df_reservas):
  df_reservas['COMPLEJO_RESERVA'] = df_reservas['HOTEL_COMPLEJO'] + ' - ' + df_reservas['NOMBRE_HABITACION']
  display(df_reservas[['HOTEL_COMPLEJO', 'NOMBRE_HABITACION', 'COMPLEJO_RESERVA']].head())
  display(df_reservas['COMPLEJO_RESERVA'].nunique())

  return df_reservas

In [ ]:
df_reservas = get_COMPLEJO_RESERVA(df_reservas)

Salen más categorías. Alguna habitación como SIG JUNIOR SUITE GV ahora está en dos categorías

In [ ]:
df_reservas[df_reservas['NOMBRE_HABITACION']=='SIG JUNIOR SUITE GV'].groupby(by=['NOMBRE_HABITACION','COMPLEJO_RESERVA']).agg( tasa_CANCELADO=('CANCELADO', lambda x: (x == 1).sum() / x.count() * 100), count=('ID_RESERVA','count')).sort_values(by='tasa_CANCELADO')

### 15.12. HABITACION_TOP

In [ ]:
def get_HABITACION_TOP(df_reservas):
  ESCOGER_PONCENTAJE = 0.003
  # Calcular la frecuencia de cada NOMBRE_HABITACION
  counts = df_reservas['NOMBRE_HABITACION'].value_counts()

  # Obtener los nombres de las habitaciones que superan el % de las reservas
  percentage_threshold = 0.01  # umbral del 1%
  percentage_threshold = ESCOGER_PONCENTAJE
  top_habitaciones = counts[counts / len(df_reservas) > percentage_threshold].index.tolist()

  # Crear la nueva columna 'HABITACION_TOP'
  df_reservas['HABITACION_TOP'] = df_reservas['NOMBRE_HABITACION'].apply(lambda x: x if x in top_habitaciones else 'OTROS')

  # Verificar la distribución de la nueva columna
  display(df_reservas['HABITACION_TOP'].nunique())
  # display(df_reservas['HABITACION_TOP'].value_counts())
  # display(df_reservas['HABITACION_TOP'].value_counts(normalize=True))

  return df_reservas

In [ ]:
df_reservas = get_HABITACION_TOP(df_reservas)

In [ ]:
plt.figure(figsize=(12, 6))

# Calculate total reservations and cancellation rate per HABITACION_TOP
cancellation_data = df_reservas.groupby('HABITACION_TOP').agg(
    TOTAL_RESERVAS=('ID_RESERVA', 'count'),
    CANCELADOS=('CANCELADO', 'sum')
).reset_index()

cancellation_data['TASA_CANCELACION'] = (cancellation_data['CANCELADOS'] / cancellation_data['TOTAL_RESERVAS']) * 100

# Sort by number of reservations in descending order for the y-axis
cancellation_data_sorted_for_plot = cancellation_data.sort_values('TOTAL_RESERVAS', ascending=False).head(30)


# 'order_labels' will be the list of room names in the desired plotting order
order_labels = cancellation_data_sorted_for_plot['HABITACION_TOP'].tolist()

ax = sns.barplot(y='HABITACION_TOP', x='TOTAL_RESERVAS', data=cancellation_data_sorted_for_plot,
                 order=order_labels, # Use the list of labels for order
                #  palette='YlOrBr_r',  # Invert the colormap to have darker for higher cancellation
                 palette='YlOrBr',
                 hue='TASA_CANCELACION', # Color by cancellation rate
                 legend=False)

# plt.title(F'{df_reservas['HABITACION_TOP'].nunique()} HABITACION_TOP por Número de Reservas (%Reservas) (Tasa Cancelación)', fontsize=14, fontweight='bold')
plt.title(F'TOP HABITACIÓN por % Reservas (Tasa Cancelación)', fontsize=14, fontweight='bold')
plt.xlabel('Número de Reservas', fontsize=12)
plt.ylabel('Tipo de Habitación', fontsize=12)
# plt.yticks([])
plt.grid(False)

for p in ax.patches:
    width = p.get_width()
    # Calculate the bar's center index on the y-axis
    # p.get_y() gives the position of the bar in data coordinates, which corresponds to its index
    bar_position_index = int(p.get_y() + p.get_height() / 2)

    # Ensure the index is within the valid range of order_labels
    if bar_position_index < len(order_labels):
        room_name = order_labels[bar_position_index]
        # Find the corresponding cancellation rate using the room_name
        tasa_cancelacion = cancellation_data_sorted_for_plot.loc[cancellation_data_sorted_for_plot['HABITACION_TOP'] == room_name, 'TASA_CANCELACION'].iloc[0]

        # Calculate tasa_reservas (percentage of total reservations)
        tasa_reservas = width / len(df_reservas) * 100

        plt.text(width + 500, p.get_y() + p.get_height()/2,
                f'{tasa_reservas:.1f}% ({tasa_cancelacion:.1f}%)',
                ha='left', va='center', fontsize=10)

    else:
        # This 'else' block handles cases where bar_position_index might be out of bounds for the last bar
        # due to floating point inaccuracies, preventing the IndexError.
        pass

plt.tight_layout()
plt.show()

### 15.13. HOTEL_HABITACION_TOP = HOTEL_COMPLEJO + HABITACION_TOP

In [ ]:
def get_HOTEL_HABITACION_TOP (df_reservas):
  df_reservas['HOTEL_HABITACION_TOP'] = df_reservas['HOTEL_COMPLEJO'] + ' - ' + df_reservas['HABITACION_TOP']
  display(df_reservas['HOTEL_HABITACION_TOP'].nunique())
  # display(df_reservas['HOTEL_HABITACION_TOP'].value_counts())
  display(df_reservas['HOTEL_HABITACION_TOP'].value_counts(normalize=True))

  return df_reservas

In [ ]:
df_reservas = get_HOTEL_HABITACION_TOP (df_reservas)

In [ ]:
plt.figure(figsize=(12, 8))

# Calculate total reservations and cancellation rate per HOTEL_HABITACION_TOP
cancellation_data = df_reservas.groupby('HOTEL_HABITACION_TOP').agg(
    TOTAL_RESERVAS=('ID_RESERVA', 'count'),
    CANCELADOS=('CANCELADO', 'sum')
).reset_index()

cancellation_data['TASA_CANCELACION'] = (cancellation_data['CANCELADOS'] / cancellation_data['TOTAL_RESERVAS']) * 100

# Sort by number of reservations in descending order for the y-axis
cancellation_data_sorted_for_plot = cancellation_data.sort_values('TOTAL_RESERVAS', ascending=False)


# 'order_labels' will be the list of room names in the desired plotting order
order_labels = cancellation_data_sorted_for_plot['HOTEL_HABITACION_TOP'].tolist()

ax = sns.barplot(y='HOTEL_HABITACION_TOP', x='TOTAL_RESERVAS', data=cancellation_data_sorted_for_plot,
                 order=order_labels, # Use the list of labels for order
                #  palette='YlOrBr_r',  # Invert the colormap to have darker for higher cancellation
                 palette='YlOrBr',
                 hue='TASA_CANCELACION', # Color by cancellation rate
                 legend=False)

plt.title(F'{df_reservas['HOTEL_HABITACION_TOP'].nunique()} HOTEL_HABITACION_TOP por Número de Reservas (%Reservas) (Tasa Cancelación)', fontsize=14, fontweight='bold')
plt.xlabel('Número de Reservas', fontsize=12)
plt.ylabel('Tipo de Habitación', fontsize=12)

for p in ax.patches:
    width = p.get_width()
    # Calculate the bar's center index on the y-axis
    # p.get_y() gives the position of the bar in data coordinates, which corresponds to its index
    bar_position_index = int(p.get_y() + p.get_height() / 2)

    # Ensure the index is within the valid range of order_labels
    if bar_position_index < len(order_labels):
        room_name = order_labels[bar_position_index]
        # Find the corresponding cancellation rate using the room_name
        tasa_cancelacion = cancellation_data_sorted_for_plot.loc[cancellation_data_sorted_for_plot['HOTEL_HABITACION_TOP'] == room_name, 'TASA_CANCELACION'].iloc[0]

        # Calculate tasa_reservas (percentage of total reservations)
        tasa_reservas = width / len(df_reservas) * 100

        plt.text(width + 500, p.get_y() + p.get_height()/2,
                 f'{int(width):,} ({tasa_reservas:.1f}%) ({tasa_cancelacion:.1f}%)',
                 ha='left', va='center', fontsize=10)
    else:
        # This 'else' block handles cases where bar_position_index might be out of bounds for the last bar
        # due to floating point inaccuracies, preventing the IndexError.
        pass

plt.tight_layout()
plt.show()

Más categorías al separar por nombre hotel, pero hay que probar si explican más las dos variables juntas

### 15.14. Limpieza de Variables con Poca Varianza (AUTORIZO)
Eliminamos la variable `AUTORIZO` ya que no aporta valor explicativo significativo para el modelo.


In [ ]:
# Al no ser una variable muy explicativa nos la quitamos de encima
if 'AUTORIZO' in df_reservas.columns:
    df_reservas.drop(columns=['AUTORIZO'], inplace=True)
    print("Columna AUTORIZO eliminada.")
else:
    print("La columna AUTORIZO no existe en el dataset.")

# Verificacion final de dimensiones
print(f"Dimensiones finales del dataset preprocesado: {df_reservas.shape}")

if 'ID_CARDEX' in df_reservas.columns:
    df_reservas.drop(columns=['ID_CARDEX'], inplace=True)
    print("Columna ID_CARDEX eliminada.")
else:
    print("La columna ID_CARDEX no existe en el dataset.")

# Verificacion final de dimensiones
print(f"Dimensiones finales del dataset preprocesado: {df_reservas.shape}")

### 15.98. Limpieza Final de `IDs` Redundantes
Eliminamos columnas técnicas (`ID_...`) que ya han sido procesadas o sustituidas por variables más explicativas.


In [ ]:
# Hacemos una lista de las columnas que queremos borrar
# - ID_TIPO: Redundante con TIPO_HABITACION / TIPO_NORMALIZADO
# - ID_HOTEL: Ya filtramos y usamos NOMBRE_HOTEL
# - ID_MONEDA: Ya normalizamos a USD
# - ID_FIDELIDAD: Ya procesado en TIENE_FIDELIDAD
# - ID_CLIENTE: Ya consolidado en CANAL_CONSOLIDADO
# - TIPO: Generalmente es el código técnico de habitación, ya tenemos NORMALIZADO

columnas_drop = ['ID_TIPO', 'ID_HOTEL', 'ID_MONEDA', 'ID_FIDELIDAD', 'ID_CLIENTE', 'TIPO', 'LEAD_TIME_CANCELATION']

# Ejecutamos el drop de la lista (solo si existen)
cols_a_borrar = [col for col in columnas_drop if col in df_reservas.columns]
if cols_a_borrar:
    df_reservas.drop(columns=cols_a_borrar, inplace=True)
    print(f"Columnas eliminadas: {cols_a_borrar}")

# Verificamos
print(f"\nColumnas restantes: {len(df_reservas.columns)}")
print(df_reservas.columns.tolist())


### 12.99. Sincronización Final de Columnas
Detectamos y eliminamos columnas redundantes (`CANCELADA` vs `CANCELADO`), copias de seguridad (`VALOR_RESERVA_ORIGINAL`) y desduplicamos columnas si existen, para alinear el dataset con la estructura objetivo.


---
## 16\. EDA Express (Resumen del Dataset Final)

Realizamos una revisión rápida del estado del dataset (`df_reservas`) tras completar todo el preprocesamiento y limpieza.


In [ ]:
# ===============================================
# 2. EDA EXPRESS (Final)
# ===============================================

print('\n----------------- 1. ESTRUCTURA DEL DATASET -----------------')
df_reservas.info()

print('\n\n----------------- 2. DETECCION DE NULOS -----------------')
nulls = df_reservas.isnull().sum()
print(nulls[nulls>0])

print('\n\n----------------- 3. ANALISIS DEL TARGET -----------------')
print("Distribución del estado de reserva:")
distribucion_target = (df_reservas['CANCELADO'].value_counts(normalize=True)*100).round(2)
print(distribucion_target)
print(f"\nTotal Cancelaciones: {df_reservas['CANCELADO'].sum()}")

print('\n\n----------------- 4. ESTADISTICAS RAPIDAS -----------------')
print(".describe de variables numericas clave:")
cols_num_interes = ['NOCHES', 'PAX', 'ADULTOS', 'NENES', 'BEBES',
                   'VALOR_RESERVA', 'ADR', 'REV_PAX', 'LEAD_TIME']
# Filtramos solo las que existen
cols_existentes = [c for c in cols_num_interes if c in df_reservas.columns]

display(df_reservas[cols_existentes].describe().T)

print("\n.describe de variables categoricas (Top valores):")
display(df_reservas.describe(include=object).T)


---
## 17\. Exportacion del Dataset Final


In [ ]:
print(f'Dimensiones finales: {df_reservas.shape}')

In [ ]:
df_reservas.columns

In [ ]:
df_reservas.columns[0:50]

In [ ]:
df_reservas.columns[50:100]

In [ ]:
PROCESSED_PATH = '/content/drive/MyDrive/TFM_Aurea_Team/01_data/processed'
os.makedirs(PROCESSED_PATH, exist_ok=True)

nombre_archivo = 'df_unificado_final_v1.csv'
ruta_completa = os.path.join(PROCESSED_PATH, nombre_archivo)

print(f'Guardando dataset en: {ruta_completa}...')
df_reservas.to_csv(ruta_completa, index=False)
print('Exportacion completada.')
print(f'Dimensiones finales: {df_reservas.shape}')


## 18\. TABLA_PRECIOS

Para la productivizaicón del prototipo, precios estimados por semana hasta 2027.

Creación de una tabla de: precio/noche (ADR), para cada habitación y PAX, semanalmente, desde el año 2022 al 2027

### AÑO_MES_LLEGADA

In [ ]:
df_reservas['LLEGADA'] = pd.to_datetime(df_reservas['LLEGADA'])
df_reservas['AÑO_LLEGADA'] = df_reservas['LLEGADA'].dt.year

display(df_reservas[['LLEGADA', 'SALIDA', 'FECHA_TOMA', 'FECHA_MOD', 'AÑO_LLEGADA']])

print(np.sort(df_reservas['AÑO_LLEGADA'].unique()))

df_reservas['AÑO_LLEGADA_SIN'] = np.sin(2 * np.pi * df_reservas['AÑO_LLEGADA'] / 12)
df_reservas['AÑO_LLEGADA_COS'] = np.cos(2 * np.pi * df_reservas['AÑO_LLEGADA'] / 12)

In [ ]:
df_reservas['AÑO_MES_LLEGADA'] = df_reservas[f'AÑO_LLEGADA'].astype(str) + '-' + \
                                    df_reservas[f'MES_LLEGADA'].astype(str).str.zfill(2)
display(df_reservas[['LLEGADA', 'AÑO_MES_LLEGADA']].head())

### ADR por habitación y fecha

In [ ]:
adr_by_hotel_month = df_reservas.groupby(['HOTEL_COMPLEJO', 'AÑO_MES_LLEGADA'])['ADR'].mean().unstack()
adr_by_hotel_month.columns = ['22-01', '22-02', '22-03', '22-04', '22-05', '22-06',
       '22-07', '22-08', '22-09', '22-10', '22-11', '22-12',
       '23-01', '23-02', '23-03', '23-04', '23-05', '23-06',
       '23-07', '23-08', '23-09', '23-10', '23-11', '23-12']
display(adr_by_hotel_month)

plt.figure(figsize=(15, 5))

# Select a few hotel complexes to plot for clarity
# You can change these selections based on what you want to visualize
hotels_to_plot = adr_by_hotel_month.index.tolist()

for hotel_complex in hotels_to_plot:
    sns.lineplot(x=adr_by_hotel_month.columns, y=adr_by_hotel_month.loc[hotel_complex], label=hotel_complex, marker='o')

plt.title('ADR Medio por Mes de Llegada en cada Hotel_Complejo', fontsize=14, fontweight='bold')
plt.xlabel('Mes de Llegada', fontsize=12)
plt.ylabel('ADR Medio', fontsize=12)
plt.xticks(range(0, 24)) # Ensure all months are shown
plt.legend(title='Hotel_Complejo', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 5))

# Filtrar HOTEL_COMPLEJO para excluir 'OTROS' si existe
df_filtered_hotel = df_reservas.copy()

# Calcular la tasa de cancelación media por mes, NOMBRE_HOTEL y HOTEL_COMPLEJO
cancelacion_media_por_hab_mes = df_filtered_hotel.groupby(['AÑO_MES_LLEGADA', 'NOMBRE_HOTEL', 'HOTEL_COMPLEJO'])['CANCELADO'].mean().reset_index()
cancelacion_media_por_hab_mes['TASA_CANCELACION'] = cancelacion_media_por_hab_mes['CANCELADO'] * 100 # Convert to percentage
cancelacion_media_por_hab_mes['AÑO_MES_LLEGADA'] = cancelacion_media_por_hab_mes['AÑO_MES_LLEGADA'].str.replace('20','')

# Excluir líneas donde la tasa de cancelación sea 0% o 100%
cancelacion_media_por_hab_mes = cancelacion_media_por_hab_mes[~((cancelacion_media_por_hab_mes['TASA_CANCELACION'] == 0)
                                                              | (cancelacion_media_por_hab_mes['TASA_CANCELACION'] == 100)
                                                              | (cancelacion_media_por_hab_mes['TASA_CANCELACION'] < 15)
                                                              | (cancelacion_media_por_hab_mes['TASA_CANCELACION'] > 48)
                                                                )]

# Crear el lineplot
sns.lineplot(
    data=cancelacion_media_por_hab_mes,
    x='AÑO_MES_LLEGADA',
    y='TASA_CANCELACION',
    hue='NOMBRE_HOTEL',  # Color por hotel
    style='HOTEL_COMPLEJO',  # Estilo de línea por tipo de habitación
    # palette='tab10',  # Usar una paleta de colores para distinguir los hoteles/complejos
    palette=[PALETA_AUREA[0]] + PALETA_AUREA[2:4],
    marker='o'  # Añadir marcadores a los puntos
)

plt.title('Tasa de Cancelación Media por Hotel y Mes de Llegada (Color por Complejo)', fontsize=14, fontweight='bold')
plt.xlabel('Mes de Llegada', fontsize=12)
plt.ylabel('Tasa de Cancelación Media (%)', fontsize=12)
plt.xticks(range(0, 24))  # Mostrar todos los meses en el eje x
plt.grid(False)
plt.legend(title='NOMBRE_HOTEL / COMPLEJO_HOTEL', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 5))

# Filtrar HOTEL_COMPLEJO para excluir 'OTROS' si existe
df_filtered_hotel = df_reservas.copy()

# Calcular la tasa de cancelación media por mes, NOMBRE_HOTEL y HOTEL_COMPLEJO
cancelacion_media_por_hab_mes = df_filtered_hotel.groupby(['AÑO_MES_LLEGADA', 'NOMBRE_HOTEL', 'HOTEL_COMPLEJO'])['CANCELADO'].mean().reset_index()
cancelacion_media_por_hab_mes['TASA_CANCELACION'] = cancelacion_media_por_hab_mes['CANCELADO'] * 100 # Convert to percentage
cancelacion_media_por_hab_mes['AÑO_MES_LLEGADA'] = cancelacion_media_por_hab_mes['AÑO_MES_LLEGADA'].str.replace('20','')

# Excluir líneas donde la tasa de cancelación sea 0% o 100%
cancelacion_media_por_hab_mes = cancelacion_media_por_hab_mes[~((cancelacion_media_por_hab_mes['TASA_CANCELACION'] == 0)
                                                              | (cancelacion_media_por_hab_mes['TASA_CANCELACION'] == 100)
                                                              | (cancelacion_media_por_hab_mes['TASA_CANCELACION'] < 15)
                                                              | (cancelacion_media_por_hab_mes['TASA_CANCELACION'] > 48)
                                                                )]

# Crear el lineplot
sns.lineplot(
    data=cancelacion_media_por_hab_mes,
    x='AÑO_MES_LLEGADA',
    y='TASA_CANCELACION',
    hue='NOMBRE_HOTEL',  # Color por hotel (complex name for legend)
    # Removed 'style' argument to simplify legend
    palette=[PALETA_AUREA[0]] + PALETA_AUREA[2:4],
    marker='o'  # Añadir marcadores a los puntos
)

plt.title('Tasa de Cancelación Media por Hotel y Mes de Llegada', fontsize=14, fontweight='bold')
plt.xlabel('Mes de Llegada', fontsize=12)
plt.ylabel('Tasa de Cancelación Media (%)', fontsize=12)
plt.xticks(ticks=range(len(cancelacion_media_por_hab_mes['AÑO_MES_LLEGADA'].unique())),
           labels=cancelacion_media_por_hab_mes['AÑO_MES_LLEGADA'].unique(),
           rotation=45, ha='right') # Show all labels and rotate for readability
plt.grid(False)

# Adjust legend position
unique_hotels_for_legend = cancelacion_media_por_hab_mes['NOMBRE_HOTEL'].unique()
plt.legend(title='Nombre del Hotel',
           loc='lower center',
           bbox_to_anchor=(0.5, -0.4),
           ncol=len(unique_hotels_for_legend)) # Arrange legend items in columns

plt.tight_layout()
plt.subplots_adjust(bottom=0.25) # Make space for the legend
plt.show()

In [ ]:
plt.figure(figsize=(16, 5))

# Filtrar HOTEL_COMPLEJO para excluir 'OTROS' si existe
df_filtered_hotel = df_reservas.copy()

# Calcular el ADR medio por mes, NOMBRE_HOTEL y HOTEL_COMPLEJO
adr_medio_por_hab_mes = df_filtered_hotel.groupby(['AÑO_MES_LLEGADA', 'NOMBRE_HOTEL', 'HOTEL_COMPLEJO'])['ADR'].mean().reset_index()
adr_medio_por_hab_mes['AÑO_MES_LLEGADA'] = adr_medio_por_hab_mes['AÑO_MES_LLEGADA'].str.replace('20','')

# Crear el lineplot
sns.lineplot(
    data=adr_medio_por_hab_mes,
    x='AÑO_MES_LLEGADA',
    y='ADR',
    hue='NOMBRE_HOTEL',  # Color por hotel
    style='HOTEL_COMPLEJO',  # Estilo de línea por tipo de habitación
    # palette='tab10',  # Usar una paleta de colores para distinguir los hoteles/complejos
    palette=[PALETA_AUREA[0]] + PALETA_AUREA[2:4],
    marker='o'  # Añadir marcadores a los puntos
)

plt.title('ADR Medio por Hotel y Mes de Llegada (Color por Complejo)')
plt.xlabel('Mes de Llegada')
plt.ylabel('ADR Medio')
plt.xticks(range(0, 24))  # Mostrar todos los meses en el eje x
plt.grid(True)
plt.legend(title='NOMBRE_HOTEL / COMPLEJO_HOTEL', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 5))

# Filtrar HABITACION_TOP para excluir 'OTROS' si existe
df_filtered_habitacion_top = df_reservas[df_reservas['HABITACION_TOP'] != 'OTROS'].copy()

# Calcular la tasa de cancelación media por mes, NOMBRE_HOTEL y HABITACION_TOP
cancelacion_media_por_hab_mes = df_filtered_habitacion_top.groupby(['AÑO_MES_LLEGADA', 'NOMBRE_HOTEL', 'HABITACION_TOP'])['CANCELADO'].mean().reset_index()
cancelacion_media_por_hab_mes['TASA_CANCELACION'] = cancelacion_media_por_hab_mes['CANCELADO'] * 100 # Convert to percentage
cancelacion_media_por_hab_mes['AÑO_MES_LLEGADA'] = cancelacion_media_por_hab_mes['AÑO_MES_LLEGADA'].str.replace('20','')

# Excluir líneas donde la tasa de cancelación sea 0% o 100%
cancelacion_media_por_hab_mes = cancelacion_media_por_hab_mes[~((cancelacion_media_por_hab_mes['TASA_CANCELACION'] == 0) | (cancelacion_media_por_hab_mes['TASA_CANCELACION'] == 100) | (cancelacion_media_por_hab_mes['TASA_CANCELACION'] < 15))]

# Crear el lineplot, suprimiendo la leyenda automática
sns.lineplot(
    data=cancelacion_media_por_hab_mes,
    x='AÑO_MES_LLEGADA',
    y='TASA_CANCELACION',
    hue='NOMBRE_HOTEL',  # Colorea por complejo
    style='HABITACION_TOP',  # Estilo de línea por tipo de habitación (para todas las líneas)
    palette=[PALETA_AUREA[0]] + PALETA_AUREA[2:4],
    marker='o',
    legend=False  # Suprimir la leyenda automática combinada
)

plt.title('Tasa de Cancelación Media por Habitación_Top y Mes de Llegada (Color por Complejo)', fontsize=14, fontweight='bold')
plt.xlabel('Mes de Llegada', fontsize=12)
plt.ylabel('Tasa de Cancelación Media (%)', fontsize=12)
plt.xticks(range(0, 24))  # Mostrar todos los meses en el eje x
plt.grid(False)

# Crear una leyenda manual solo para los complejos
unique_complexes = cancelacion_media_por_hab_mes['NOMBRE_HOTEL'].unique()
colors = sns.color_palette(palette=[PALETA_AUREA[0]] + PALETA_AUREA[2:4]) # Usar la misma paleta

handles = []
labels = []
for i, complex_name in enumerate(unique_complexes):
    handles.append(plt.Line2D([0], [0], color=colors[i], lw=2)) # Línea dummy para la leyenda
    labels.append(complex_name)

plt.legend(handles=handles, labels=labels, title='Hotel_Complejo', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 5))

# Filtrar HABITACION_TOP para excluir 'OTROS' si existe
df_filtered_habitacion_top = df_reservas[df_reservas['HABITACION_TOP'] != 'OTROS'].copy()

# Calcular el ADR medio por mes, NOMBRE_HOTEL y HABITACION_TOP
adr_medio_por_hab_mes = df_filtered_habitacion_top.groupby(['AÑO_MES_LLEGADA', 'NOMBRE_HOTEL', 'HABITACION_TOP'])['ADR'].mean().reset_index()
adr_medio_por_hab_mes['AÑO_MES_LLEGADA'] = adr_medio_por_hab_mes['AÑO_MES_LLEGADA'].str.replace('20','')

# Crear el lineplot, suprimiendo la leyenda automática
sns.lineplot(
    data=adr_medio_por_hab_mes,
    x='AÑO_MES_LLEGADA',
    y='ADR',
    hue='NOMBRE_HOTEL',  # Colorea por complejo
    style='HABITACION_TOP',  # Estilo de línea por tipo de habitación (para todas las líneas)
    # palette='tab10',  # Usar una paleta de colores
    palette=[PALETA_AUREA[0]] + PALETA_AUREA[2:4],
    marker='o',
    legend=False  # Suprimir la leyenda automática combinada
)

plt.title('ADR Medio por Habitación_Top y Mes de Llegada (Color por Complejo)', fontsize=14, fontweight='bold')
plt.xlabel('Mes de Llegada', fontsize=12)
plt.ylabel('ADR Medio', fontsize=12)
plt.xticks(range(0, 24))  # Mostrar todos los meses en el eje x
plt.grid(True)

# Crear una leyenda manual solo para los complejos
unique_complexes = adr_medio_por_hab_mes['NOMBRE_HOTEL'].unique()
# colors = sns.color_palette('tab10', n_colors=len(unique_complexes)) # Usar la misma paleta
colors = sns.color_palette(palette=[PALETA_AUREA[0]] + PALETA_AUREA[2:4]) # Usar la misma paleta

handles = []
labels = []
for i, complex_name in enumerate(unique_complexes):
    handles.append(plt.Line2D([0], [0], color=colors[i], lw=2)) # Línea dummy para la leyenda
    labels.append(complex_name)

plt.legend(handles=handles, labels=labels, title='Hotel_Complejo', loc='lower center', bbox_to_anchor=(0.5, -0.3), ncol=len(unique_complexes))

plt.tight_layout()
plt.subplots_adjust(bottom=0.25)
plt.show()

In [ ]:
ESCOGER_PONCENTAJE=0.003

# Calcular la frecuencia de cada COMPLEJO_RESERVA_TOP
counts = df_reservas['COMPLEJO_RESERVA'].value_counts()

# Obtener los nombres de las habitaciones que superan el % de las reservas
percentage_threshold = 0.01  # umbral del 1%
percentage_threshold = ESCOGER_PONCENTAJE
top_habitaciones = counts[counts / len(df_reservas) > percentage_threshold].index.tolist()

# Crear la nueva columna 'COMPLEJO_RESERVA_TOP'
df_reservas['COMPLEJO_RESERVA_TOP'] = df_reservas['COMPLEJO_RESERVA'].apply(lambda x: x if x in top_habitaciones else 'OTROS')

# Verificar la distribución de la nueva columna
display(df_reservas['COMPLEJO_RESERVA_TOP'].nunique())

date_cols = ['LLEGADA']
for col in date_cols:
    df_reservas[col] = pd.to_datetime(df_reservas[col])

# Crear las nuevas columnas para cada fecha
for col in date_cols:
    df_reservas[f'AÑO_{col}'] = df_reservas[col].dt.year


In [ ]:
date_cols = ['LLEGADA']

for col in date_cols:
    # Get the year and week number
    year = df_reservas[col].dt.year.astype(str)
    week = df_reservas[col].dt.isocalendar().week.astype(str).str.zfill(2)
    df_reservas[f'AÑO_SEMANA_{col}'] = year + '-' + week

In [ ]:
display(df_reservas.groupby(['HOTEL_COMPLEJO', 'COMPLEJO_RESERVA_TOP', 'PAX', 'AÑO_SEMANA_LLEGADA'])['ADR'].mean().unstack(level=['AÑO_SEMANA_LLEGADA']).head(15))

In [ ]:
display(df_reservas.groupby(['HOTEL_COMPLEJO', 'COMPLEJO_RESERVA_TOP', 'PAX', 'AÑO_LLEGADA', 'MES_LLEGADA'])['ADR'].mean().unstack(level=['AÑO_LLEGADA', 'MES_LLEGADA']).head(15))

### Ceación de TABLA_PRECIOS

In [ ]:
date_cols = ['LLEGADA']

for col in date_cols:
    # Get the year and week number
    year = df_reservas[col].dt.year.astype(str)
    week = df_reservas[col].dt.isocalendar().week.astype(str).str.zfill(2)
    df_reservas[f'AÑO_SEMANA_{col}'] = year + '-' + week

In [ ]:
#ADR mediana por habitación y PAX, para cada semana
TABLA_PRECIOS_ADR_habitacion_PAX_semana = df_reservas.groupby(['HOTEL_COMPLEJO', 'COMPLEJO_RESERVA_TOP', 'PAX', 'AÑO_SEMANA_LLEGADA'])['ADR'].mean().unstack(level=['AÑO_SEMANA_LLEGADA'])
TABLA_PRECIOS_ADR_habitacion_PAX_semana = TABLA_PRECIOS_ADR_habitacion_PAX_semana.reset_index(col_level=0)
TABLA_PRECIOS_ADR_habitacion_PAX_semana.columns.name = None # Remove the index name

In [ ]:
TABLA_PRECIOS_ADR_habitacion_PAX_semana

In [ ]:
# #ADR mediana por habitación y PAX, para cada semana
# TABLA_PRECIOS_ADR_habitacion_PAX_semana = df_reservas.groupby(['HOTEL_COMPLEJO', 'COMPLEJO_RESERVA_TOP', 'PAX', 'AÑO_LLEGADA', 'SEMANA_AÑO_LLEGADA'])['ADR'].mean().unstack(level=['AÑO_LLEGADA', 'SEMANA_AÑO_LLEGADA'])
# TABLA_PRECIOS_ADR_habitacion_PAX_semana.columns = [f'{col[0]}-{col[1]}' for col in TABLA_PRECIOS_ADR_habitacion_PAX_semana.columns]
# TABLA_PRECIOS_ADR_habitacion_PAX_semana = TABLA_PRECIOS_ADR_habitacion_PAX_semana.reset_index(col_level=0)

2024

Riviera maya creció el precio 6% con respecto al 2023

Costa Mujeres creció el precio 2% con respecto al 2023

Punta Cana creció el precio un 2% frente al 2023


In [ ]:
# Define the growth rates for each hotel complex for 2024
growth_rates = {
    'MAYA': 1.06,  # Riviera Maya
    'MUJE': 1.02,  # Costa Mujeres
    'CANA': 1.02   # Punta Cana
}

# Get all columns for 2023
columns_2023 = [col for col in TABLA_PRECIOS_ADR_habitacion_PAX_semana.columns if col.startswith('2023-')]

# Pre-allocate new columns for 2024 to avoid fragmentation warnings
new_2024_cols = [f'2024-{col.split("-")[1]}' for col in columns_2023]
TABLA_PRECIOS_ADR_habitacion_PAX_semana[new_2024_cols] = np.nan

# Add new columns for 2024
for col_2023 in columns_2023:
    week = col_2023.split('-')[1]
    col_2024 = f'2024-{week}'

    TABLA_PRECIOS_ADR_habitacion_PAX_semana[col_2024] = np.nan # Initialize with NaN

    for complex_prefix, rate in growth_rates.items():
        mask = TABLA_PRECIOS_ADR_habitacion_PAX_semana['HOTEL_COMPLEJO'].str.startswith(complex_prefix)
        TABLA_PRECIOS_ADR_habitacion_PAX_semana.loc[mask, col_2024] = \
            TABLA_PRECIOS_ADR_habitacion_PAX_semana.loc[mask, col_2023] * rate

display(TABLA_PRECIOS_ADR_habitacion_PAX_semana.head())
display(TABLA_PRECIOS_ADR_habitacion_PAX_semana.filter(like='2023').head())
display(TABLA_PRECIOS_ADR_habitacion_PAX_semana.filter(like='2024').head())

2025

Riviera maya creció el precio 1% con respecto al 2024

Costa Mujeres creció el precio 1% con respecto al 2024

Punta Cana Decrecio el precio un 3% frente al 2024


In [ ]:
# Define the growth rates for each hotel complex for 2025
growth_rates = {
    'MAYA': 1.01,  # Riviera Maya
    'MUJE': 1.01,  # Costa Mujeres
    'CANA': 0.97   # Punta Cana
}

# Get all columns for 2024
columns_2024 = [col for col in TABLA_PRECIOS_ADR_habitacion_PAX_semana.columns if col.startswith('2024-')]

# Create a temporary DataFrame for 2025 prices with the same index as the main DataFrame
df_2025_prices = pd.DataFrame(index=TABLA_PRECIOS_ADR_habitacion_PAX_semana.index)

# Calculate and fill 2025 prices into the temporary DataFrame
for col_2024 in columns_2024:
    week = col_2024.split('-')[1]
    col_2025 = f'2025-{week}'

    # Initialize the column in the temporary DataFrame
    df_2025_prices[col_2025] = np.nan

    for complex_prefix, rate in growth_rates.items():
        mask = TABLA_PRECIOS_ADR_habitacion_PAX_semana['HOTEL_COMPLEJO'].str.startswith(complex_prefix)
        df_2025_prices.loc[mask, col_2025] = \
            TABLA_PRECIOS_ADR_habitacion_PAX_semana.loc[mask, col_2024] * rate

# Concatenate the new 2025 columns to the main DataFrame once
TABLA_PRECIOS_ADR_habitacion_PAX_semana = pd.concat([TABLA_PRECIOS_ADR_habitacion_PAX_semana, df_2025_prices], axis=1)

display(TABLA_PRECIOS_ADR_habitacion_PAX_semana.head())
display(TABLA_PRECIOS_ADR_habitacion_PAX_semana.filter(like='2024').head())
display(TABLA_PRECIOS_ADR_habitacion_PAX_semana.filter(like='2025').head())

2026 y 2027

Para realizar una estimación de precios para 2026 y 2027, debemos considerar que la industria turística en el Caribe está entrando en una fase de normalización tras el "boom" post-pandemia.
Basado en los datos históricos que tenemos y en las proyecciones actuales del sector (donde se observa una desaceleración en el ritmo de crecimiento en México y una recuperación estratégica en República Dominicana), hacemos una propuesta de estimación:

·

Resumen de Estimaciones (%)

Destino ············ 2024 (Real) ···· 2025 (Real) ···· 2026 (Est.) ····2027 (Est.)

Riviera Maya ········ +6%  ··············· +1%  ············· +3% a +4% ········· +2.5%

Costa Mujeres ······ +2% ·············· +1% ············· +2% a +3% ············ +2%

Punta Cana ··········· +2% ·············· -3% ············· +1% a +2% ············· +3%

·

Análisis por Destino

1. Riviera Maya

Contexto: Tras un 2025 muy plano (+1%), el 2026 estará marcado por el Mundial de la FIFA 2026. Aunque los partidos no son en el Caribe, el flujo masivo de turistas a México genera un efecto de "derrame" hacia los destinos de playa.

Proyección: Se espera una recuperación moderada.
El crecimiento no volverá al 6% de 2024 debido a la competencia de otros mercados, pero el factor eventos y la inflación proyectada impulsarán un alza de entre 3% y 4%.
Para 2027, el precio debería estabilizarse cerca del 2.5%.

2. Costa Mujeres

Contexto: Es un destino más exclusivo y "nuevo", lo que le da una estabilidad mayor. Su crecimiento ha sido más conservador pero constante.

Proyección: Al ser una zona de lujo con oferta controlada, no suele sufrir caídas drásticas. Se estima un crecimiento inercial del 2% al 3% anual para ambos años, alineado con la inflación hotelera estándar.

3. Punta Cana

Contexto: El 2025 fue un año de ajuste (deflación del -3%), probablemente por una sobreoferta o una corrección tras precios muy altos en 2024.

Proyección: Tras una caída, suele venir un rebote técnico. Grandes cadenas (como Meliá o Posadas) han anunciado planes de expansión y subidas de tarifas cercanas al 5% para sus segmentos premium en 2026. Sin embargo, para el promedio general del destino, una estimación conservadora es volver a terreno positivo (1% a 2%) en 2026 y recuperar un ritmo de crucero del 3% en 2027.

.

Factores a considerar

Efecto Mundial 2026: México tendrá una visibilidad internacional máxima, lo que suele permitir a los hoteleros subir tarifas base.

Recuperación de la Demanda: El Banco Mundial proyecta que el crecimiento económico en el Caribe se acelere hacia 2027 (llegando al 3.7% excluyendo Guyana), lo que respalda un aumento en los precios de servicios turísticos.

In [ ]:
# Define the growth rates for each hotel complex for 2026
growth_rates = {
    'MAYA': 1.03,  # Riviera Maya
    'MUJE': 1.02,  # Costa Mujeres
    'CANA': 1.01   # Punta Cana
}

# Get all columns for 2025
columns_2025 = [col for col in TABLA_PRECIOS_ADR_habitacion_PAX_semana.columns if col.startswith('2025-')]

# Create a temporary DataFrame for 2026 prices with the same index as the main DataFrame
df_2026_prices = pd.DataFrame(index=TABLA_PRECIOS_ADR_habitacion_PAX_semana.index)

# Calculate and fill 2026 prices into the temporary DataFrame
for col_2025 in columns_2025:
    week = col_2025.split('-')[1]
    col_2026 = f'2026-{week}'

    # Initialize the column in the temporary DataFrame
    df_2026_prices[col_2026] = np.nan

    for complex_prefix, rate in growth_rates.items():
        mask = TABLA_PRECIOS_ADR_habitacion_PAX_semana['HOTEL_COMPLEJO'].str.startswith(complex_prefix)
        df_2026_prices.loc[mask, col_2026] = \
            TABLA_PRECIOS_ADR_habitacion_PAX_semana.loc[mask, col_2025] * rate

# Concatenate the new 2026 columns to the main DataFrame once
TABLA_PRECIOS_ADR_habitacion_PAX_semana = pd.concat([TABLA_PRECIOS_ADR_habitacion_PAX_semana, df_2026_prices], axis=1)

display(TABLA_PRECIOS_ADR_habitacion_PAX_semana.head())
display(TABLA_PRECIOS_ADR_habitacion_PAX_semana.filter(like='2025').head())
display(TABLA_PRECIOS_ADR_habitacion_PAX_semana.filter(like='2026').head())

In [ ]:
# Define the growth rates for each hotel complex for 2027
growth_rates = {
    'MAYA': 1.025,  # Riviera Maya
    'MUJE': 1.02,  # Costa Mujeres
    'CANA': 1.03   # Punta Cana
}

# Get all columns for 2026
columns_2026 = [col for col in TABLA_PRECIOS_ADR_habitacion_PAX_semana.columns if col.startswith('2026-')]

# Create a temporary DataFrame for 2027 prices with the same index as the main DataFrame
df_2027_prices = pd.DataFrame(index=TABLA_PRECIOS_ADR_habitacion_PAX_semana.index)

# Calculate and fill 2027 prices into the temporary DataFrame
for col_2026 in columns_2026:
    week = col_2026.split('-')[1]
    col_2027 = f'2027-{week}'

    # Initialize the column in the temporary DataFrame
    df_2027_prices[col_2027] = np.nan

    for complex_prefix, rate in growth_rates.items():
        mask = TABLA_PRECIOS_ADR_habitacion_PAX_semana['HOTEL_COMPLEJO'].str.startswith(complex_prefix)
        df_2027_prices.loc[mask, col_2027] = \
            TABLA_PRECIOS_ADR_habitacion_PAX_semana.loc[mask, col_2026] * rate

# Concatenate the new 2027 columns to the main DataFrame once
TABLA_PRECIOS_ADR_habitacion_PAX_semana = pd.concat([TABLA_PRECIOS_ADR_habitacion_PAX_semana, df_2027_prices], axis=1)

display(TABLA_PRECIOS_ADR_habitacion_PAX_semana.head())
display(TABLA_PRECIOS_ADR_habitacion_PAX_semana.filter(like='2026').head())
display(TABLA_PRECIOS_ADR_habitacion_PAX_semana.filter(like='2027').head())

In [ ]:
TABLA_PRECIOS_ADR_habitacion_PAX_semana

In [ ]:
PROCESSED_PATH = '/content/drive/MyDrive/TFM_Aurea_Team/01_data/processed'
os.makedirs(PROCESSED_PATH, exist_ok=True)

nombre_archivo = 'TABLA_PRECIOS_ADR_habitacion_PAX_semana.csv'
ruta_completa = os.path.join(PROCESSED_PATH, nombre_archivo)

print(f'Guardando dataset en: {ruta_completa}...')
TABLA_PRECIOS_ADR_habitacion_PAX_semana.to_csv(ruta_completa, index=False)
print('Exportacion completada.')
print(f'Dimensiones finales: {TABLA_PRECIOS_ADR_habitacion_PAX_semana.shape}')

In [ ]:
TABLA_PRECIOS_ADR_habitacion_PAX_semana.head()

In [ ]:
TABLA_PRECIOS_ADR_habitacion_PAX_semana.tail()

In [ ]:
TABLA_PRECIOS_ADR_habitacion_PAX_semana.columns

In [ ]:
TABLA_PRECIOS_ADR_habitacion_PAX_semana['COMPLEJO_RESERVA_TOP'].nunique()

In [ ]:
TABLA_PRECIOS_ADR_habitacion_PAX_semana['COMPLEJO_RESERVA_TOP'].unique()

---
## 19\. Librería de funciones para productivización

Funciones para pipeline de transformación de las variables para predecir con el modelo


In [ ]:
# Creamos la funciones necesarias para construir las variables a partir de los datos primitivos existentes en las observaciones
import inspect

# a partir de los datos en campos sueltos
def get_features(LLEGADA, NOCHES, PAX, ADULTOS, CLIENTE, FECHA_TOMA, FIDELIDAD, PAIS, SEGMENTO, FUENTE_NEGOCIO, NOMBRE_HOTEL, NOMBRE_HABITACION, VALOR_RESERVA):
  data = {
      'LLEGADA': [LLEGADA],
      'NOCHES': [NOCHES],
      'PAX': [PAX],
      'ADULTOS': [ADULTOS],
      'CLIENTE': [CLIENTE],
      'ID_MULTIPLE': 0,
      'FECHA_TOMA': [FECHA_TOMA],
      'FIDELIDAD': [FIDELIDAD],
      'PAIS': [PAIS],
      'SEGMENTO': [SEGMENTO],
      'FUENTE_NEGOCIO': [FUENTE_NEGOCIO],
      'NOMBRE_HOTEL': [NOMBRE_HOTEL],
      'NOMBRE_HABITACION': [NOMBRE_HABITACION],
      'VALOR_RESERVA': [VALOR_RESERVA]
  }

  df_reservas_in = pd.DataFrame(data)

  df_reservas_in['LLEGADA'] = pd.to_datetime(df_reservas_in['LLEGADA'])
  df_reservas_in['FECHA_TOMA'] = pd.to_datetime(df_reservas_in['FECHA_TOMA'])

  df_reservas_in = get_ADR(df_reservas_in)
  df_reservas_in = get_REV_PAX(df_reservas_in)
  df_reservas_in = get_LEAD_TIME(df_reservas_in)
  df_reservas_in['PAIS_TOP_200'] = df_reservas_in['PAIS']
  df_reservas_in = get_HORA_TOMA(df_reservas_in)
  df_reservas_in = get_TIPO_VIAJERO(df_reservas_in)
  df_reservas_in = get_LOYALTY_REWARDS_y_TIENE_FIDELIDAD(df_reservas_in)
  df_reservas_in = get_ES_GRUPO(df_reservas_in)
  df_reservas_in = get_MES_LLEGADA(df_reservas_in)
  df_reservas_in = get_NOMBRE_COMPLEJO(df_reservas_in)
  df_reservas_in = get_PREFIJO_HAB(df_reservas_in)
  df_reservas_in = get_HOTEL_COMPLEJO(df_reservas_in)
  df_reservas_in = get_COMPLEJO_RESERVA(df_reservas_in)
  df_reservas_in = get_HABITACION_TOP(df_reservas_in)
  df_reservas_in = get_HOTEL_HABITACION_TOP (df_reservas_in)
  df_reservas_in = get_FUENTE_NEGOCIO_SEGMENTO_CLIENTE (df_reservas_in)

  return df_reservas_in


# a partir de un dataframe con reservas
def get_features_df(df_reservas_in):

  df_reservas_in['LLEGADA'] = pd.to_datetime(df_reservas_in['LLEGADA'])
  df_reservas_in['FECHA_TOMA'] = pd.to_datetime(df_reservas_in['FECHA_TOMA'])

  df_reservas_in = get_ADR(df_reservas_in)
  df_reservas_in = get_REV_PAX(df_reservas_in)
  df_reservas_in = get_LEAD_TIME(df_reservas_in)
  df_reservas_in['PAIS_TOP_200'] = df_reservas_in['PAIS']
  df_reservas_in = get_HORA_TOMA(df_reservas_in)
  df_reservas_in = get_TIPO_VIAJERO(df_reservas_in)
  df_reservas_in = get_LOYALTY_REWARDS_y_TIENE_FIDELIDAD(df_reservas_in)
  df_reservas_in = get_ES_GRUPO(df_reservas_in)
  df_reservas_in = get_MES_LLEGADA(df_reservas_in)
  df_reservas_in = get_NOMBRE_COMPLEJO(df_reservas_in)
  df_reservas_in = get_PREFIJO_HAB(df_reservas_in)
  df_reservas_in = get_HOTEL_COMPLEJO(df_reservas_in)
  df_reservas_in = get_COMPLEJO_RESERVA(df_reservas_in)
  df_reservas_in = get_HABITACION_TOP(df_reservas_in)
  df_reservas_in = get_HOTEL_HABITACION_TOP (df_reservas_in)
  df_reservas_in = get_FUENTE_NEGOCIO_SEGMENTO_CLIENTE (df_reservas_in)

  return df_reservas_in


# Gruardar las funciones en un fichero Python .py en Google Drive
file_path = '/content/drive/MyDrive/TFM_Aurea_Team/funciones_produccion.py'

with open(file_path, 'w') as f:
    # Añadimos los imports necesarios al principio de fichero .py
    f.write('import pandas as pd\n')
    f.write('import numpy as np\n')
    f.write('import yfinance as yf\n')

    f.write(inspect.getsource(get_ADR))
    f.write(inspect.getsource(get_REV_PAX))
    f.write(inspect.getsource(get_LEAD_TIME))
    f.write(inspect.getsource(get_PAIS_TOP_200))
    f.write(inspect.getsource(get_HORA_TOMA))
    f.write(inspect.getsource(get_TIPO_VIAJERO))
    f.write(inspect.getsource(get_LOYALTY_REWARDS_y_TIENE_FIDELIDAD))
    f.write(inspect.getsource(get_ES_GRUPO))
    f.write(inspect.getsource(get_MES_LLEGADA))
    f.write(inspect.getsource(get_NOMBRE_COMPLEJO))
    f.write(inspect.getsource(get_PREFIJO_HAB))
    f.write(inspect.getsource(get_HOTEL_COMPLEJO))
    f.write(inspect.getsource(get_COMPLEJO_RESERVA))
    f.write(inspect.getsource(get_HABITACION_TOP))
    f.write(inspect.getsource(get_HOTEL_HABITACION_TOP))
    f.write(inspect.getsource(get_FUENTE_NEGOCIO_SEGMENTO_CLIENTE))

    f.write(inspect.getsource(get_features_df))
    f.write(inspect.getsource(get_features))

print(f'Funciones grabadas en {file_path}')

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/TFM_Aurea_Team/')

from funciones_produccion import get_features

In [ ]:
# Example values for a reservation (replace with your actual data)
LLEGADA = '2023-11-06'
NOCHES = 8
PAX = 2
ADULTOS = 2
CLIENTE = 'AUREA TRAVEL CLUB_SOCIOS'
FECHA_TOMA = '2023-02-22 12:48:29'
FIDELIDAD = None
PAIS = 'ESTADOS UNIDOS'
SEGMENTO = 'Fixed rates'
FUENTE_NEGOCIO = 'DIRECT SALES'
NOMBRE_HOTEL = 'Complejo Riviera Maya'
NOMBRE_HABITACION = 'SIG Junior Suite PS'
VALOR_RESERVA = 2305.60

# Call the function
new_reservation_df = get_features(
    LLEGADA, NOCHES, PAX, ADULTOS, CLIENTE, FECHA_TOMA, FIDELIDAD, PAIS, SEGMENTO, FUENTE_NEGOCIO, NOMBRE_HOTEL, NOMBRE_HABITACION, VALOR_RESERVA
)

# Display the resulting DataFrame
print(new_reservation_df)

---
## 20\. Análisis Univariante tras FE

Analizamos la distribución individual de las variables numéricas y categóricas más relevantes, así como la variable objetivo (`CANCELADO`).


### 20.1. Variables Numéricas (Distribución y Outliers)
Visualizamos Histogramas y Boxplots para `ADR`, `VALOR_RESERVA`, `LEAD_TIME`, `NOCHES` y `REV_PAX`.


In [ ]:
vars_nums = ['HORA_TOMA', 'MES_LLEGADA']

# Configuración Estética
sns.set_style("whitegrid")
plt.figure(figsize=(18, 5 * len(vars_nums)))

for i, var in enumerate(vars_nums):
    # Histograma (Distribución)
    plt.subplot(len(vars_nums), 2, 2*i + 1)
    sns.histplot(df_reservas[var], kde=True, color=PALETA_AUREA[0], bins=50)
    plt.title(f'Distribución de {var}', fontsize=12, fontweight='bold')
    plt.xlabel(var)
    plt.ylabel('Frecuencia')

    # Boxplot (Outliers)
    plt.subplot(len(vars_nums), 2, 2*i + 2)
    sns.boxplot(x=df_reservas[var], color=PALETA_AUREA[1])
    plt.title(f'Boxplot de {var}', fontsize=12, fontweight='bold')
    plt.xlabel(var)

plt.tight_layout()
plt.show()

### 20.2. Variables Categóricas (Rankings)
Analizamos las principales categorías de negocio: `CANAL_CONSOLIDADO`, `TIPO_VIAJERO`, `SEGMENTO`, `TIPO_NORMALIZADO` y `REGIMEN`.


In [ ]:
vars_cats = df_reservas.select_dtypes(object).columns
vars_cats

In [ ]:
vars_cats =['TIPO_VIAJERO', 'COMPLEJO_RESERVA']

In [ ]:
# plt.figure(figsize=(18, 6 * len(vars_cats)))

for i, var in enumerate(vars_cats):
    plt.figure(figsize=(12, 14))

    plt.subplot(len(vars_cats), 1, i + 1)

    data = df_reservas
    # Get the order by frequency for the current variable
    order = df_reservas[var].value_counts().index

    # If there are more than 15 unique values, filter `data` to include only the top 15
    if df_reservas[var].nunique() > 15:
        top_15_values = order[:15]
        data = df_reservas[df_reservas[var].isin(top_15_values)]
        order = top_15_values

    # Generate an extended palette to ensure explicit cycling for all categories
    num_colors_needed = len(order)
    # Calculate how many times the base palette needs to be repeated
    num_repeats = (num_colors_needed // len(PALETA_AUREA)) + 1
    # Create an extended palette by repeating PALETA_AUREA and slicing to the exact number of categories
    extended_palette = (PALETA_AUREA * num_repeats)[:num_colors_needed]

    # Grafico de barras
    ax = sns.countplot(y=var, data=data, order=order, palette=extended_palette, hue=var, legend=False)
    plt.title(f'Frecuencia por {var}', fontsize=14, fontweight='bold')
    plt.xlabel('Número de Reservas')
    plt.ylabel(var)

    # Etiquetas de datos
    for p in ax.patches:
        width = p.get_width()
        plt.text(width + 500, p.get_y() + p.get_height()/2,
                 '{:1.0f}'.format(width),
                 ha='left', va='center')

    plt.tight_layout()
    plt.show()

---
## 21\. Analisis Bivariante tras FE

Cruces de variables nuevas contra el target CANCELADO.

### 21.1. Categoricas vs Target


In [ ]:
vars_cats =['TIPO_VIAJERO',
       'ES_GRUPO', 'LOYALTY_REWARDS', 'TIENE_FIDELIDAD',
        'COMPLEJO_RESERVA', 'HOTEL_HABITACION_TOP']

In [ ]:
vars_cats_bi = vars_cats

In [ ]:
for var in vars_cats_bi:
    if var in df_reservas.columns:

        data = df_reservas.copy()

        top = ''

        # Si hay más de 15 categorías, filtramos el top 15
        if df_reservas[var].nunique() > 10:
          top_15_values = df_reservas[var].value_counts().head(10).index
          data = df_reservas[df_reservas[var].isin(top_15_values)].copy()
          top = 'TOP '

        ct = pd.crosstab(data[var], data['CANCELADO'], normalize='index') * 100
        ct = ct.sort_values(by=1, ascending=False)

        ax = ct.plot(kind='barh', stacked=True, figsize=(12, 6), color=['#D28B57', '#99782C'])
        plt.title(f'Tasa de Cancelacion por {top}{var}')
        plt.ylabel('Porcentaje (%)')
        plt.xlabel(var)
        plt.legend(['Confirmada', 'Cancelada'])
        plt.tight_layout()
        plt.show()

In [ ]:
# for var in vars_cats_bi:

#     data = df_reservas.copy()

#     # Si hay más de 15 categorías, filtramos el top 15
#     if df_reservas[var].nunique() > 15:
#       top_15_values = df_reservas[var].value_counts().head(15).index
#       data = df_reservas[df_reservas[var].isin(top_15_values)].copy()

#     if var in df_reservas.columns:
#         ct = pd.crosstab(data[var], data['CANCELADO'], normalize='index') * 100
#         ct = ct.sort_values(by=1, ascending=False)

#         ax = ct.plot(kind='bar', stacked=True, figsize=(10, 5), color=['#D28B57', '#99782C'])
#         plt.title(f'Tasa de Cancelacion por {var}')
#         plt.ylabel('Porcentaje (%)')
#         plt.xlabel(var)
#         plt.legend(['Confirmada', 'Cancelada'])
#         plt.tight_layout()
#         plt.show()

### 21.2. Numericas vs Target


In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_reservas[df_reservas['HORA_TOMA']!=0], x='HORA_TOMA', hue='CANCELADO', fill=True, palette={0: COLOR_NO_CANCELADO, 1: COLOR_CANCELADO})
plt.title('Distribución de HORA_TOMA (sin 00h) por CANCELADO')
plt.xlabel('HORA_TOMA')
plt.ylabel('Densidad')
plt.legend(title='CANCELADO', labels=['Cancelado', 'No Cancelado'])
plt.tick_params(axis='y', left=False, labelleft=False) # Hide y-axis ticks and labels
plt.xlim(0.1, 24.5)
plt.tight_layout()
plt.grid(False)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_reservas, x='MES_LLEGADA', hue='CANCELADO', fill=True, palette={0: COLOR_NO_CANCELADO, 1: COLOR_CANCELADO})
plt.title('Distribución de Densidad de MES_LLEGADA por Estado de Reserva')
plt.xlabel('MES_LLEGADA')
plt.ylabel('Densidad')
plt.legend(title='CANCELADO', labels=['Cancelado', 'No Cancelado'])
plt.tick_params(axis='y', left=False, labelleft=False) # Hide y-axis ticks and labels
plt.tight_layout()
plt.grid(False)
plt.show()

In [ ]:
vars_nums = ['NOCHES', 'PAX', 'ADULTOS', 'NENES', 'BEBES',
       'SUPLETORIA',
       'CUNAS',
       'VALOR_RESERVA', 'ADR', 'REV_PAX', 'LEAD_TIME', 'HORA_TOMA',
       'MES_LLEGADA']

In [ ]:
vars_nums_bi = vars_nums

### 21.3. Matriz de Correlacion


In [ ]:
vars_nums = ['NOCHES', 'PAX', 'ADULTOS', 'NENES', 'BEBES',
       'SUPLETORIA',
       'CUNAS',
       'VALOR_RESERVA', 'ADR', 'REV_PAX', 'LEAD_TIME', 'HORA_TOMA',
       'MES_LLEGADA']

In [ ]:
cols_corr = vars_nums

In [ ]:
# cols_corr = ['ADR', 'LEAD_TIME', 'NOCHES', 'PAX', 'VALOR_RESERVA', 'REV_PAX', 'CANCELADO']
cols_corr = [c for c in cols_corr if c in df_reservas.columns]

plt.figure(figsize=(10, 8))
corr_matrix = df_reservas[cols_corr].corr()
sns.heatmap(corr_matrix, annot=True, cmap='YlOrBr', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlacion')
plt.tight_layout()
plt.show()


### 21.4. Estacionalidad


In [ ]:
tasa_mes = df_reservas.groupby('HORA_TOMA')['CANCELADO'].mean() * 100

plt.figure(figsize=(12, 6))
ax = sns.barplot(x=tasa_mes.index, y=tasa_mes.values, color='#6E550C')
plt.title('Tasa de Cancelacion por Hora de Toma (excluida la hora 00)')
plt.xlabel('Hora')
plt.ylabel('Tasa de Cancelacion (%)')
plt.ylim(0, 60)

for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + 1, f'{height:.1f}%', ha='center')

plt.tight_layout()
plt.show()